In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockA — Hindcast O3 (30–70 hPa polar-cap column) scores vs reference
=====================================================================

This script does the following:

1. Experiment design & naming
   --------------------------
   Hindcast output directory (already created by your shell script):

       /home/weiji/restart_exam/hindcast_data/
           YYYY-MM* /
               O3/      <-- ONLY this variable is used here
                   *.O3.nc
               U/
               T/
               VTH3d/

   Case naming convention:

       - Directory starts with "YYYY-MM":
           * "YYYY" is the model year label (e.g. "0008").
           * "MM" is the initialization month (01 = Jan, 02 = Feb, ...),
             and the hindcast is initialized on day 1 of that month.

       - After "YYYY-MM" there may be a suffix:

           (1) ""   (no suffix)
               -> small-perturbation, coupled O3.
                  ~30 members where the initial TEMPERATURE fields are
                  perturbed with extremely small random noise of order
                  1e-12 K.
                  We call these groups:
                      <Month>_couple

           (2) "V2"
               -> large-perturbation ensemble.
                  ~30 members where the initial TEMPERATURE fields are
                  perturbed with much larger random noise of order
                  1e-6 K.
                  Group name:
                      <Month>_largePert

           (3) "V3"
               -> Q-perturbation ensemble.
                  ~30 members where the initial WATER VAPOUR fields are
                  perturbed, but the pattern is constructed so that total
                  water vapour is (approximately) conserved: regions with
                  increased humidity are compensated by regions with
                  reduced humidity.
                  Group name:
                      <Month>_Qpert

           (4) "nocouple" (e.g. "YYYY-MM-nocouple")
               -> XX_initial_prescribeO3 ensemble.
                  Same small TEMPERATURE perturbations as the small-pert
                  ensemble, but O3 is prescribed (no ozone coupling).
                  Group name:
                      <Month>_initial_prescribeO3

   In this BlockA we:
       * Load ONLY O3 from each hindcast case.
       * Classify each case into a family:
             "small_perturbation"
             "large_perturbation"
             "q_perturbation"
             "initial_prescribed_o3"
       * Construct a human-readable group label:
             "Feb_couple", "Feb_initial_prescribeO3", etc.

2. Reference simulation
   ---------------------
   The reference ("truth") is:

       /mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/
           BWCN.e122.f19_g16.merged.nc

   We read its O3, compute:
       - 30–70 hPa partial column in DU, and
       - 60–90N polar-cap mean.

   The hindcast diagnostics (30–70 hPa polar-cap column) are computed in
   exactly the same way, so that they are directly comparable.

3. What this BlockA actually computes
   ----------------------------------
   - We focus only on YEAR 0008 hindcasts (case directories whose name
     starts with "0008-").
   - For each such case (e.g. "0008-02", "0008-02V2", "0008-02-nocouple"):

       (a) Load O3(member, time, plev_2, lat, lon).
       (b) Compute 30–70 hPa partial column (DU) and 60–90N polar-cap
           mean, giving O3_cap(member, time).
       (c) Align the time axis with the reference polar-cap column
           O3_ref(time).
       (d) For each day of the hindcast (from the initialization date):
               - CRPS (continuous ranked probability score) vs reference
               - ensemble spread (standard deviation across members)
               - mean error (bias = ensemble mean - reference)
               - mean absolute error (optional, stored as well)

       (e) Save a NetCDF file for each case:

           /home/weiji/test_code/plots/hindcast/
               O3_scores_{case_name}.nc

           with variables:
               - O3_cap(member, time)
               - O3_ref(time)
               - CRPS(time)
               - spread(time)
               - bias(time)
               - mae(time)
             and metadata including group label, family, etc.

   This gives you daily time series of CRPS / spread / mean error for
   each experiment group (e.g. "Feb_couple" for year 0008), ready to be
   plotted as in the paper.
"""

import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import xarray as xr


# ---------------------------------------------------------------------
# Paths & constants
# ---------------------------------------------------------------------

HINDCAST_BASE_DIR = "/home/weiji/restart_exam/hindcast_data"
REF_O3_PATH = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc"
ROOT_OUT_DIR = "/home/weiji/test_code/plots/hindcast/O3"

os.makedirs(ROOT_OUT_DIR, exist_ok=True)




# ---------------------------------------------------------------------
# Experiment family metadata
# ---------------------------------------------------------------------

MONTH_NAME = {
    "01": "Jan",
    "02": "Feb",
    "03": "Mar",
    "04": "Apr",
    "05": "May",
    "06": "Jun",
    "07": "Jul",
    "08": "Aug",
    "09": "Sep",
    "10": "Oct",
    "11": "Nov",
    "12": "Dec",
}

BASE_FAMILIES: Dict[str, Dict[str, str]] = {
    "": {
        "family": "small_perturbation",
        "short_name": "SP",
        "description": (
            "Small-perturbation ensemble (coupled O3): ~30 members where the "
            "initial temperature fields are perturbed with extremely small "
            "random noise of order 1e-12 K."
        ),
    },
    "V2": {
        "family": "large_perturbation",
        "short_name": "LP",
        "description": (
            "Large-perturbation ensemble (V2): ~30 members where the initial "
            "temperature fields are perturbed with much larger random noise "
            "of order 1e-6 K."
        ),
    },
    "V3": {
        "family": "q_perturbation",
        "short_name": "QP",
        "description": (
            "Q-perturbation ensemble (V3): ~30 members where the initial "
            "water vapour fields are perturbed in a mass-conserving way "
            "(regions with enhanced humidity are compensated by regions "
            "with reduced humidity)."
        ),
    },
}


# ---------------------------------------------------------------------
# Dataclass: one hindcast O3 case
# ---------------------------------------------------------------------

@dataclass
class O3HindcastCase:
    """Container for one hindcast case (O3 ensemble only)."""

    case_dir: Path           # /.../hindcast_data/YYYY-MM*
    case_name: str           # e.g. "0008-02", "0008-02V2", "0008-02-nocouple"
    year_str: str            # "0008"
    month_str: str           # "02"
    tag: str                 # "", "V2", "V3", "-nocouple", ...
    family: str              # small_perturbation / large_perturbation / q_perturbation / initial_prescribed_o3 / unknown
    family_short: str        # SP / LP / QP / NC / UNK
    description: str         # human-readable description
    group_label: str         # e.g. "Feb_couple", "Feb_initial_prescribeO3"
    n_member: int            # number of ensemble members
    member_ids: np.ndarray   # e.g. [1, 2, ..., 30]
    o3: xr.DataArray         # dims: (member, time, plev_2 or lev/plev, lat, lon)


# ---------------------------------------------------------------------
# Parsing & classification of case names (incl. nocouple)
# ---------------------------------------------------------------------

def parse_case_name(case_name: str) -> Dict[str, str]:
    """
    Parse directory name like "0008-02", "0008-02V2", "0008-02V3", "0008-02-nocouple".

    Returns:
        {
            "year_str", "month_str", "tag",
            "family", "family_short", "description", "group_label"
        }
    """
    m = re.match(r"^(?P<year>\d{4})-(?P<month>\d{2})(?P<suffix>.*)$", case_name)
    if not m:
        raise ValueError(f"Case name does not match 'YYYY-MM*' pattern: {case_name}")

    year_str = m.group("year")
    month_str = m.group("month")
    raw_suffix = m.group("suffix")  # may be "", "V2", "V3", "-nocouple", "nocouple", ...

    tag = raw_suffix

    # Default: unknown family
    family = "unknown"
    family_short = "UNK"
    description = (
        f"Unknown perturbation family (suffix='{raw_suffix}') for case '{case_name}'. "
        "Treat as a generic ensemble."
    )

    # Exact matches: "", "V2", "V3"
    if raw_suffix in BASE_FAMILIES:
        meta = BASE_FAMILIES[raw_suffix]
        family = meta["family"]
        family_short = meta["short_name"]
        description = meta["description"]

    # Nocouple: any suffix containing "nocouple"
    elif "nocouple" in raw_suffix.lower():
        family = "initial_prescribed_o3"
        family_short = "NC"
        description = (
            "Nocouple ensemble: same small temperature perturbations as the "
            "small-pert ensemble, but ozone is prescribed (no ozone coupling). "
            "This corresponds to XX_initial_prescribeO3."
        )

    month_label = MONTH_NAME.get(month_str, month_str)

    if family == "small_perturbation" and "nocouple" not in raw_suffix.lower():
        group_label = f"{month_label}_couple"
    elif family == "initial_prescribed_o3":
        group_label = f"{month_label}_initial_prescribeO3"
    elif family == "large_perturbation":
        group_label = f"{month_label}_largePert"
    elif family == "q_perturbation":
        group_label = f"{month_label}_Qpert"
    else:
        group_label = case_name  # fallback if unknown

    return dict(
        year_str=year_str,
        month_str=month_str,
        tag=tag,
        family=family,
        family_short=family_short,
        description=description,
        group_label=group_label,
    )


# ---------------------------------------------------------------------
# Hindcast O3 loaders
# ---------------------------------------------------------------------

def list_case_dirs(base_dir: str | Path) -> List[Path]:
    """List all hindcast case directories under base_dir whose name matches 'YYYY-MM*'."""
    base_dir = Path(base_dir).expanduser().resolve()
    case_dirs: List[Path] = []
    for p in sorted(base_dir.iterdir()):
        if not p.is_dir():
            continue
        if re.match(r"^\d{4}-\d{2}.*$", p.name):
            case_dirs.append(p)
    return case_dirs


def list_o3_files(case_dir: Path) -> List[Path]:
    """List all O3 NetCDF files for a given case (case_dir / 'O3' / '*.nc')."""
    o3_dir = case_dir / "O3"
    if not o3_dir.is_dir():
        return []
    return sorted(o3_dir.glob("*.nc"))


def parse_member_ids_from_filenames(files: List[Path], case_name: str) -> np.ndarray:
    """
    Parse member IDs from filenames.

    Example filename:
        BWCN.e122.f19_g16.007.0008-02.001.small-pertlim.cam.h3.O3.nc
        parts = ["BWCN","e122","f19_g16","007","0008-02","001","small-pertlim",...]

    Strategy:
      - Find the token equal to 'case_name' in the split list.
      - The next token, if purely digits, is treated as the member index.
      - If parsing fails, fall back to 1..N.
    """
    member_ids: List[int] = []

    for idx_file, f in enumerate(files):
        parts = f.name.split(".")
        mem_id: int | None = None

        if case_name in parts:
            i = parts.index(case_name)
            if i + 1 < len(parts) and parts[i + 1].isdigit():
                try:
                    mem_id = int(parts[i + 1])
                except Exception:
                    mem_id = None

        if mem_id is None:
            mem_id = idx_file + 1  # fallback: 1..N

        member_ids.append(mem_id)

    return np.asarray(member_ids, dtype=int)


def load_o3_for_case(case_dir: Path) -> O3HindcastCase | None:
    """
    为单个 hindcast case 读取所有 O3 成员，并统一垂直维到 'plev'，
    避免 open_mfdataset 因为部分文件是 plev / 部分是 plev_2 而合成 6 维怪物。

    返回一个 O3HindcastCase，o3 的维度应为：
        (member, time, plev, lat, lon)
    """
    case_name = case_dir.name
    meta = parse_case_name(case_name)

    files = list_o3_files(case_dir)
    if not files:
        print(f"[INFO] No O3 files found in case {case_name}, skipping.")
        return None

    files = sorted(files)
    member_ids = parse_member_ids_from_filenames(files, case_name)
    n_member = len(files)

    print(f"[DEBUG] (new loader) Opening O3 for case {case_name}, members={n_member}")

    da_list = []

    for f, mid in zip(files, member_ids):
        ds_single = xr.open_dataset(f, decode_times=True)
        if "O3" not in ds_single.data_vars:
            print(f"[WARN] File {f} has no O3, skip this member.")
            ds_single.close()
            continue

        da = ds_single["O3"]

        # ---- 统一垂直维名字：优先使用 plev ----
        dims = da.dims

        if ("plev_2" in dims) and ("plev" not in dims):
            # 典型 hindcast 输出：time, plev_2, lat, lon
            da = da.rename({"plev_2": "plev"})
        elif ("plev" in dims) and ("plev_2" not in dims):
            # 已经是我们想要的名字
            pass
        elif ("plev" in dims) and ("plev_2" in dims):
            # 极少见：单个文件就带两个垂直维，如果发生，就提示一下并做一个保守处理
            print(f"[WARN] File {f} has both 'plev' and 'plev_2'; "
                  f"temporarily dropping 'plev_2' dimension by taking index 0.")
            da = da.isel(plev_2=0, drop=True)

        ds_single.close()  # 先关文件，再用内存中的 DataArray

        # ---- 增加 member 维度并赋上对应的 member id ----
        da = da.expand_dims("member")
        da = da.assign_coords(member=[mid])

        da_list.append(da)

    if len(da_list) == 0:
        print(f"[WARN] No valid O3 DataArray for case {case_name}, skip.")
        return None

    # ---- 沿 member 维拼接所有成员 ----
    da_o3 = xr.concat(da_list, dim="member")

    print(f"[DEBUG] Combined O3 dims for {case_name}: "
          f"{da_o3.dims}, sizes={dict(da_o3.sizes)}")

    # 构造 case 对象
    case = O3HindcastCase(
        case_dir=case_dir,
        case_name=case_name,
        year_str=meta["year_str"],
        month_str=meta["month_str"],
        tag=meta["tag"],
        family=meta["family"],
        family_short=meta["family_short"],
        description=meta["description"],
        group_label=meta["group_label"],
        n_member=da_o3.sizes["member"],
        member_ids=da_o3["member"].values,
        o3=da_o3,
    )
    return case



def load_all_o3_hindcasts(base_dir: str | Path) -> Tuple[List[O3HindcastCase], Dict[str, List[O3HindcastCase]]]:
    """
    Scan base_dir for all hindcast cases, load O3 ensembles,
    and group them by experiment family.
    """
    base_dir = Path(base_dir).expanduser().resolve()
    all_cases: List[O3HindcastCase] = []
    by_family: Dict[str, List[O3HindcastCase]] = {}

    case_dirs = list_case_dirs(base_dir)
    print(f"[INFO] Found {len(case_dirs)} candidate case directories under {base_dir}")

    for case_dir in case_dirs:
        case = load_o3_for_case(case_dir)
        if case is None:
            continue

        all_cases.append(case)
        fam = case.family
        by_family.setdefault(fam, []).append(case)

        # 打印基本信息（包含实验组名称）
        print(
            f"[LOADED] {case.case_name:18s}  "
            f"family={case.family_short:3s}  "
            f"group={case.group_label:25s}  "
            f"members={case.n_member:2d}"
        )

    print("[INFO] Finished loading O3 hindcasts.")
    for fam, lst in by_family.items():
        print(f"  - {fam:22s}: {len(lst)} cases")

    return all_cases, by_family


# ---------------------------------------------------------------------
# 30–70 hPa partial column O3 (DU) & polar-cap mean
# （基于你给的 calc_parc_o3 + 60–90N 极盖平均）
# ---------------------------------------------------------------------

def calc_parc_o3_du(var, p_top=30, p_bottom=70):
    """
    计算 p_top–p_bottom hPa 的部分臭氧柱（DU）。

    这是你原来的 calc_parc_o3(p_top, p_bottom) 逻辑，
    只是额外支持 'plev_2' 作为垂直坐标名，并可以带 member 维。
    """
    m_air = 28.964 / (6.022E23)
    g = 980.6

    # ---- 识别垂直维 ----
    if 'plev' in var.dims:
        plev = var.plev; level = 'plev'
    elif 'plev_2' in var.dims:
        plev = var.plev_2; level = 'plev_2'
    elif 'lev' in var.dims:
        plev = var.lev; level = 'lev'
    elif 'level' in var.dims:
        plev = var.level; level = 'level'
    else:
        raise ValueError('No pressure level coordinate found in data.')

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    # ---- 判定单位 & 升/降序 ----
    if plev[0] < plev[-1] and plev[-1] <= 1000:
        factor, factor_2 = 100, 1
    elif plev[0] > plev[-1] and plev[0] <= 1000:
        factor, factor_2 = 100, 1
    elif plev[0] < plev[-1] and plev[-1] > 1000:
        factor, factor_2 = 1, 100
    else:
        factor, factor_2 = 1, 100

    # ---- 升序 ----
    if plev[0] < plev[-1]:
        for i in range(1, len(plev)):
            delta_p[:, i] = plev[i] - plev[i-1]

        weights_p = xr.DataArray(
            delta_p * factor,
            dims=['time', level],
            coords=[time, plev]
        )
        O3 = var * weights_p * 10 / (g * m_air)

        O3 = O3.sel(**{level: slice(p_top * factor_2, p_bottom * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16

    # ---- 降序 ----
    else:
        for i in range(len(plev) - 1):
            delta_p[:, i] = plev[i] - plev[i+1]

        weights_p = xr.DataArray(
            delta_p * factor,
            dims=['time', level],
            coords=[time, plev]
        )
        O3 = var * weights_p * 10 / (g * m_air)

        O3 = O3.sel(**{level: slice(p_bottom * factor_2, p_top * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16

    return O3.where(O3 != 0)



def polar_cap_mean_30_70(o3_4d: xr.DataArray,
                         lat_min: float = 60.0,
                         lat_max: float = 90.0) -> xr.DataArray:
    """
    给定 O3 四维场（可以包含 member 维），计算：
      1) 沿 lon 做经向平均（如果有 lon 维）
      2) 30–70 hPa 部分臭氧柱（DU） —— 使用上面的 calc_parc_o3_du
      3) 60–90N 极盖余弦加权平均

    返回：
      - 若有 member 维：dims = (member, time)
      - 若无 member 维：dims = (time,)
    """

    # 1. 先经向平均（完全照你旧代码）
    if 'lon' in o3_4d.dims:
        o3_zm = o3_4d.mean(dim='lon')
    else:
        o3_zm = o3_4d

    # 2. 若垂直维叫 plev_2，则统一改名为 plev，让积分函数少操心
    if 'plev_2' in o3_zm.dims and 'plev' not in o3_zm.dims:
        o3_zm = o3_zm.rename({'plev_2': 'plev'})

    # 3. 计算 30–70 hPa 部分柱（DU）
    o3_pc = calc_parc_o3_du(o3_zm, p_top=30, p_bottom=70)
    # 此时维度应该是 (member?, time, lat)
    print("[DEBUG] after calc_parc_o3_du dims:", o3_pc.dims, dict(o3_pc.sizes))

    # 4. 60–90N 极盖平均（cos(lat) 加权）
    o3_cap = o3_pc.sel(lat=slice(lat_min, lat_max))
    weights = np.cos(np.deg2rad(o3_cap.lat))
    o3_cap = o3_cap.weighted(weights).mean(dim='lat')

    print(f"[DEBUG] polar_cap_mean_30_70: output dims={o3_cap.dims}, sizes={dict(o3_cap.sizes)}")
    return o3_cap

# ---------------------------------------------------------------------
# CRPS / spread / mean error 子程序
# ---------------------------------------------------------------------

def crps_ensemble(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    """
    CRPS for a discrete ensemble:

        CRPS = 1/m * sum |Fi - o| - 1/(2 m^2) * sum_{i,j} |Fi - Fj|

    ensemble: shape (m, ...)
    obs     : shape (...) (broadcastable)

    返回：
        shape (...) — 每个时间点的 CRPS。
    """
    ens = np.asarray(ensemble)
    obs_arr = np.asarray(obs)

    if ens.ndim < 1:
        raise ValueError("ensemble must have at least 1 dimension: (member, ...)")

    diff_to_obs = np.abs(ens - obs_arr)
    term1 = diff_to_obs.mean(axis=0)

    diff_ij = np.abs(ens[:, None, ...] - ens[None, :, ...])
    term2 = 0.5 * diff_ij.mean(axis=(0, 1))

    return term1 - term2


def ensemble_spread(ensemble: np.ndarray) -> np.ndarray:
    """成员标准差（沿 member 维）."""
    ens = np.asarray(ensemble)
    return ens.std(axis=0, ddof=0)


def ensemble_bias(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    """Ensemble mean minus observation: mean(F) - o."""
    ens = np.asarray(ensemble)
    obs_arr = np.asarray(obs)
    return ens.mean(axis=0) - obs_arr


def ensemble_mae(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    """Mean absolute error across members: mean(|Fi - o|)."""
    ens = np.asarray(ensemble)
    obs_arr = np.asarray(obs)
    return np.abs(ens - obs_arr).mean(axis=0)


# ---------------------------------------------------------------------
# Reference O3 loader (30–70 hPa polar-cap column)
# ---------------------------------------------------------------------

def load_ref_o3_polarcap(ref_path: str | Path = REF_O3_PATH) -> xr.DataArray:
    """
    从 BWCN merged 文件中读取 O3，并计算 30–70 hPa 极盖平均（60–90N）臭氧柱（DU）。

    返回：
        DataArray: dims = ('time',)，单位 DU。
    """
    ref_path = Path(ref_path).expanduser().resolve()
    print(f"[REF] Opening reference file: {ref_path}")

    ds_ref = xr.open_dataset(ref_path, decode_times=True)
    print(f"[REF] Reference dims: {dict(ds_ref.dims)}")

    if "O3" not in ds_ref.data_vars:
        raise KeyError(f"O3 not found in reference dataset. Vars: {list(ds_ref.data_vars)}")

    o3_ref = ds_ref["O3"]
    # 同样改成 dims + sizes
    print(f"[REF] O3 dims: {o3_ref.dims}, sizes={dict(o3_ref.sizes)}")
    print(f"[REF] time range: {o3_ref.time.min().values} -> {o3_ref.time.max().values}")

    col_ref = polar_cap_mean_30_70(o3_ref)  # dims: (time,)
    col_ref.name = "O3_pc_60_90N_30_70hPa"

    return col_ref


# ---------------------------------------------------------------------
# 计算单个 hindcast case 的 CRPS / spread / bias
# ---------------------------------------------------------------------

def compute_scores_for_case(case: O3HindcastCase,
                            ref_col: xr.DataArray) -> xr.Dataset:
    """
    对单个 hindcast case（一个初始化、一个实验组）计算：

        - O3_cap(member, time) : 30–70 hPa 极盖柱（DU）
        - O3_ref(time)         : 参考极盖柱（DU）
        - CRPS(time)
        - spread(time)
        - bias(time)
        - mae(time)

    返回一个 xr.Dataset，用于保存到 NetCDF。
    """
    print(f"\n[CASE] Processing case: {case.case_name} ({case.group_label})")
    print(f"       family={case.family}  members={case.n_member}")

    # 1. 计算 hindcast 的极盖柱
    o3_cap = polar_cap_mean_30_70(case.o3)  # dims: (member, time)
    print(f"[DEBUG] O3_cap dims: {o3_cap.dims}, sizes={dict(o3_cap.sizes)}")
    print(f"[DEBUG] O3_cap time range: {o3_cap.time.min().values} -> {o3_cap.time.max().values}")

    # 2. 与参考对齐时间轴（inner 交集）
    o3_cap_aligned, ref_aligned = xr.align(o3_cap, ref_col, join="inner")
    print(f"[DEBUG] After align: hindcast time len={o3_cap_aligned.time.size}, "
          f"ref time len={ref_aligned.time.size}")

    if o3_cap_aligned.time.size == 0:
        raise RuntimeError(f"No overlapping time between hindcast {case.case_name} and reference.")

    # 3. 转 numpy 计算评分
    ens_np = o3_cap_aligned.values  # (member, time)
    obs_np = ref_aligned.values     # (time,)

    crps = crps_ensemble(ens_np, obs_np)     # (time,)
    spread = ensemble_spread(ens_np)         # (time,)
    bias = ensemble_bias(ens_np, obs_np)     # (time,)
    mae = ensemble_mae(ens_np, obs_np)       # (time,)

    # 4. 生成 lead（预报天数，从 0 开始）
    n_time = o3_cap_aligned.time.size
    lead = np.arange(n_time, dtype=int)

    # 5. 组装 Dataset
    ds_out = xr.Dataset(
        coords={
            "time": o3_cap_aligned.time,
            "lead": ("time", lead),
            "member": o3_cap_aligned.member,
        },
        data_vars={
            "O3_cap": (("member", "time"), ens_np),
            "O3_ref": (("time",), obs_np),
            "CRPS": (("time",), crps),
            "spread": (("time",), spread),
            "bias": (("time",), bias),
            "mae": (("time",), mae),
        },
        attrs={
            "case_name": case.case_name,
            "year_str": case.year_str,
            "month_str": case.month_str,
            "group_label": case.group_label,
            "family": case.family,
            "family_short": case.family_short,
            "description": case.description,
            "note": (
                "30–70 hPa polar-cap (60–90N) O3 column in DU; "
                "scores (CRPS, spread, bias, mae) computed daily "
                "from hindcast start, relative to reference BWCN "
                "merged simulation."
            ),
        },
    )

    # 添加单位
    ds_out["O3_cap"].attrs["units"] = "DU"
    ds_out["O3_ref"].attrs["units"] = "DU"
    ds_out["CRPS"].attrs["units"] = "DU"
    ds_out["spread"].attrs["units"] = "DU"
    ds_out["bias"].attrs["units"] = "DU"
    ds_out["mae"].attrs["units"] = "DU"

    return ds_out


# ---------------------------------------------------------------------
# BlockA 主函数：只处理 year=0008 的所有 case
# ---------------------------------------------------------------------

def main_blockA():
    print("[BlockA] Starting hindcast O3 BlockA ...")
    print(f"[BlockA] Hindcast base dir: {HINDCAST_BASE_DIR}")
    print(f"[BlockA] Reference file   : {REF_O3_PATH}")
    print(f"[BlockA] Output root      : {ROOT_OUT_DIR}")

    # 1. 读取所有 hindcast O3 case，并按 family 归类
    cases, by_family = load_all_o3_hindcasts(HINDCAST_BASE_DIR)

    # 2. 只保留 year=0008 的 case
    cases_0008 = [c for c in cases if c.year_str == "0008"]
    print(f"\n[BlockA] Found {len(cases_0008)} hindcast cases for year 0008.")
    for c in cases_0008:
        print(f"         - {c.case_name:18s}  group={c.group_label:25s}  family={c.family_short}")

    if not cases_0008:
        print("[BlockA] No year 0008 cases found. Nothing to do.")
        return

    # 3. 读取参考 30–70 hPa 极盖柱
    ref_col = load_ref_o3_polarcap(REF_O3_PATH)  # dims: (time,)
    print(f"[BlockA] Reference polar-cap O3 time len={ref_col.time.size}")
    print(f"[BlockA] Reference time range: {ref_col.time.min().values} -> {ref_col.time.max().values}")

    # 4. 对每一个 year 0008 的 case 计算 scores 并写输出
    for case in cases_0008:
        ds_scores = compute_scores_for_case(case, ref_col)

        out_path = os.path.join(ROOT_OUT_DIR, f"O3_scores_{case.case_name}.nc")
        ds_scores.to_netcdf(out_path)
        print(f"[BlockA] Saved scores for {case.case_name} to: {out_path}")

    print("\n[BlockA] All year-0008 hindcast cases processed. Done.")


if __name__ == "__main__":
    main_blockA()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB — O3 验证指标可视化（CRPS / mean error / ensemble spread）
#
# 依赖：
#   * BlockA 已经运行，生成：
#       /home/weiji/test_code/plots/hindcast/O3_scores_*.nc
#   * O3_scores_*.nc 中包含变量：
#       - CRPS   (time)
#       - bias   (time)  —— ensemble mean - reference
#       - spread (time)  —— ensemble std dev
#
# 输出：
#   对每个 group、每个指标（metric）：
#     1) lead-time 图（X 轴 lead day 1–150）
#     2) calendar 图（X 轴 Jan–May：Jan/Feb/Mar/Apr/May）
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ----------------- 基本路径与参数 -----------------
SCORES_ROOT_DIR = "/home/weiji/test_code/plots/hindcast/O3"
FIG_OUT_DIR     = SCORES_ROOT_DIR

LEAD_START_DEFAULT = 1
LEAD_END_DEFAULT   = 150   # 统一画 1–150 天

# 针对不同指标的默认 y 轴范围（可以以后按需要调）
Y_LIM_DEFAULTS = {
    "CRPS":   (0.0, 50.0),
    "bias":   (-50.0, 50.0),   # mean error 一般上下对称
    "spread": (0.0, 50.0),
}

# 指标配置：变量名、漂亮名字、y 轴标签
METRICS = [
    {
        "var_name": "CRPS",
        "pretty":   "CRPS",
        "ylabel":   "CRPS (polar-cap 30–70 hPa O$_3$) [DU]",
    },
    {
        "var_name": "bias",
        "pretty":   "Mean error",
        "ylabel":   "Mean error (ensemble mean $-$ reference) [DU]",
    },
    {
        "var_name": "spread",
        "pretty":   "Ensemble spread",
        "ylabel":   "Ensemble spread (std across members) [DU]",
    },
]

# ----------------- 分组配置（与你原来的保持一致） -----------------
GROUPS = [
    # 组0：small-pert，不同 initial month
    {
        "group_id": 0,
        "name": "group0_small_different_init_month",
        "title": "Group 0 – small perturbation, different initialization month",
        "cases": [
            {"case_name": "0008-01", "label": "Jan small (chem) 0008-01", "color": "C0"},
            {"case_name": "0008-02", "label": "Feb small (chem) 0008-02", "color": "C1"},
            {"case_name": "0008-03", "label": "Mar small (chem) 0008-03", "color": "C2"},
        ],
    },
    # 组1：Feb chem vs nocouple
    {
        "group_id": 1,
        "name": "group1_Feb_chem_vs_nochem",
        "title": "Group 1 – Feb initialization: chem vs prescribe-O3",
        "cases": [
            {"case_name": "0008-02",         "label": "Feb chem (0008-02)",         "color": "C0"},
            {"case_name": "0008-02_NOCOUPL", "label": "Feb prescribe-O3 (NOCOUPL)", "color": "C3"},
        ],
    },
    # 组2：Mar chem vs nocouple
    {
        "group_id": 2,
        "name": "group2_Mar_chem_vs_nochem",
        "title": "Group 2 – Mar initialization: chem vs prescribe-O3",
        "cases": [
            {"case_name": "0008-03",         "label": "Mar chem (0008-03)",         "color": "C0"},
            {"case_name": "0008-03_NOCOUPL", "label": "Mar prescribe-O3 (NOCOUPL)", "color": "C3"},
        ],
    },
    # 组3：large-pert，不同 initial month
    {
        "group_id": 3,
        "name": "group3_largepert_different_init_month",
        "title": "Group 3 – large perturbation, different initialization month",
        "cases": [
            {"case_name": "0008-02_v2", "label": "Feb large-pert (0008-02_v2)", "color": "C0"},
            {"case_name": "0008-03_v2", "label": "Mar large-pert (0008-03_v2)", "color": "C1"},
            {"case_name": "0008-04_v2", "label": "Apr large-pert (0008-04_v2)", "color": "C2"},
        ],
    },
    # 组4：Mar init，不同 pert
    {
        "group_id": 4,
        "name": "group4_Mar_different_pert",
        "title": "Group 4 – Mar initialization, different perturbations",
        "cases": [
            {"case_name": "0008-03",    "label": "Mar small-pert (0008-03)",    "color": "C0"},
            {"case_name": "0008-03_v2", "label": "Mar large-pert (0008-03_v2)", "color": "C1"},
            {"case_name": "0008-03_v3", "label": "Mar Q-pert (0008-03_v3)",     "color": "C2"},
        ],
    },
    # 组5：Q-pert，不同 init month
    {
        "group_id": 5,
        "name": "group5_Qpert_different_init_month",
        "title": "Group 5 – Q-pert, different initialization month",
        "cases": [
            {"case_name": "0008-02_v3", "label": "Feb Q-pert (0008-02_v3)", "color": "C0"},
            {"case_name": "0008-03_v3", "label": "Mar Q-pert (0008-03_v3)", "color": "C1"},
        ],
    },
    # 组6：Feb init，不同 pert
    {
        "group_id": 6,
        "name": "group6_Feb_different_pert",
        "title": "Group 6 – Feb initialization, different perturbations",
        "cases": [
            {"case_name": "0008-02",    "label": "Feb small-pert (0008-02)",    "color": "C0"},
            {"case_name": "0008-02_v2", "label": "Feb large-pert (0008-02_v2)", "color": "C1"},
            {"case_name": "0008-02_v3", "label": "Feb Q-pert (0008-02_v3)",     "color": "C2"},
        ],
    },
]

# ============================================================
# 一、lead-time 版本（对任意 metric）
# ============================================================

def load_metric_lead_window(
    case_name: str,
    var_name: str,
    lead_start: int = LEAD_START_DEFAULT,
    lead_end:   int = LEAD_END_DEFAULT,
    root_dir:   str = SCORES_ROOT_DIR,
) -> xr.DataArray | None:
    """
    从 O3_scores_{case_name}.nc 读取指定变量 var_name，并截取 lead day 窗口。
    lead_start / lead_end 为 1-based（1 = 初始化当天）。
    """
    nc_path = os.path.join(root_dir, f"O3_scores_{case_name}.nc")
    if not os.path.exists(nc_path):
        print(f"[BlockB] WARNING: File not found for {case_name}: {nc_path}")
        return None

    ds = xr.open_dataset(nc_path)
    if var_name not in ds.data_vars:
        print(
            f"[BlockB] WARNING: Variable '{var_name}' not in {nc_path}. "
            f"Available: {list(ds.data_vars)}"
        )
        ds.close()
        return None

    da = ds[var_name].sortby("time")  # dims: ('time',)
    n_time = da.sizes["time"]
    print(f"[BlockB] {case_name}: {var_name} total length = {n_time} days")
    print(f"         time range = {da.time.min().values} -> {da.time.max().values}")
    ds.close()

    # 处理 lead 窗口
    if lead_start < 1:
        print(f"[BlockB]  lead_start < 1 for {case_name}, clamp to 1.")
        lead_start = 1
    if lead_end < lead_start:
        print(f"[BlockB]  ERROR: lead_end({lead_end}) < lead_start({lead_start}) for {case_name}")
        return None
    if lead_start > n_time:
        print(f"[BlockB]  WARNING: lead_start({lead_start}) > n_time({n_time}) for {case_name}; skip.")
        return None

    eff_end = min(lead_end, n_time)   # 不能超过实际长度
    idx_start = lead_start - 1        # 0-based
    idx_end   = eff_end               # slice 上界不包含

    da_sel = da.isel(time=slice(idx_start, idx_end))
    print(
        f"[BlockB] {case_name}: {var_name} using lead days {lead_start}–{eff_end} "
        f"(indices {idx_start}..{eff_end-1}), n={da_sel.sizes['time']}"
    )
    return da_sel


def plot_group_metric_lead(
    group_cfg: dict,
    metric_cfg: dict,
    scores_root: str = SCORES_ROOT_DIR,
    fig_dir:     str = FIG_OUT_DIR,
    lead_start:  int = LEAD_START_DEFAULT,
    lead_end:    int = LEAD_END_DEFAULT,
):
    """为一个 group + 指标 画 lead-time 图。"""
    os.makedirs(fig_dir, exist_ok=True)

    gid    = group_cfg["group_id"]
    gname  = group_cfg["name"]
    gtitle = group_cfg["title"]
    cases  = group_cfg["cases"]

    var_name = metric_cfg["var_name"]
    pretty   = metric_cfg["pretty"]
    ylabel   = metric_cfg["ylabel"]
    y_lim    = Y_LIM_DEFAULTS.get(var_name, None)

    print(f"\n[BlockB] ==== Lead-time group {gid}, metric={var_name} ====")

    series = []
    for cfg in cases:
        case_name = cfg["case_name"]
        da = load_metric_lead_window(
            case_name=case_name,
            var_name=var_name,
            lead_start=lead_start,
            lead_end=lead_end,
            root_dir=scores_root,
        )
        if da is None:
            continue

        n_use = da.sizes["time"]
        eff_start = lead_start
        eff_end   = lead_start + n_use - 1
        series.append((cfg, da, eff_start, eff_end))

    if not series:
        print(f"[BlockB]  No valid series for group {gid}, metric={var_name}, skip.")
        return

    fig, ax = plt.subplots(figsize=(8, 5))

    for cfg, da, eff_start, eff_end in series:
        n_use = da.sizes["time"]
        lead_days = np.arange(eff_start, eff_start + n_use)
        ax.plot(
            lead_days,
            da.values,
            label=cfg["label"],
            color=cfg.get("color", None),
            linewidth=2,
        )

    ax.set_xlabel("Lead time (days since initialization)", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(
        gtitle + f"\nMetric: {pretty} (lead days {lead_start}–{lead_end})",
        fontsize=13,
    )

    ax.set_xlim(lead_start, lead_end)
    if y_lim is not None:
        ax.set_ylim(y_lim[0], y_lim[1])
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)
    fig.tight_layout()

    base_name = f"{var_name}_lead_group{gid}_{gname}_days{lead_start}_{lead_end}"
    out_png = os.path.join(fig_dir, base_name + ".png")
    out_pdf = os.path.join(fig_dir, base_name + ".pdf")
    fig.savefig(out_png, dpi=300)
    fig.savefig(out_pdf, dpi=300)


def main_blockB_lead():
    print("[BlockB] Start lead-time plots for all metrics ...")
    for metric_cfg in METRICS:
        var_name = metric_cfg["var_name"]
        print(f"\n[BlockB] === Metric: {var_name} ===")
        for g in GROUPS:
            plot_group_metric_lead(
                group_cfg=g,
                metric_cfg=metric_cfg,
                scores_root=SCORES_ROOT_DIR,
                fig_dir=FIG_OUT_DIR,
                lead_start=LEAD_START_DEFAULT,
                lead_end=LEAD_END_DEFAULT,
            )
    print("[BlockB] Lead-time plots done.")

# ============================================================
# 二、calendar-time 版本（Jan–May，对任意 metric）
# ============================================================

def load_metric_until_end_of_may(
    case_name: str,
    var_name: str,
    root_dir: str = SCORES_ROOT_DIR,
) -> xr.DataArray | None:
    """
    从 O3_scores_{case_name}.nc 中读取指定变量，并裁剪到
    “该年的 1 月 1 日 ~ 5 月 31 日”。
    """
    nc_path = os.path.join(root_dir, f"O3_scores_{case_name}.nc")
    if not os.path.exists(nc_path):
        print(f"[BlockB-2] WARNING: file not found for {case_name}: {nc_path}")
        return None

    ds = xr.open_dataset(nc_path)
    if var_name not in ds.data_vars:
        print(
            f"[BlockB-2] WARNING: variable '{var_name}' not in {nc_path}. "
            f"Available: {list(ds.data_vars)}"
        )
        ds.close()
        return None

    da = ds[var_name].sortby("time")
    ds.close()

    months = da.time.dt.month
    days   = da.time.dt.day
    years  = da.time.dt.year

    ref_year = int(years.values[0])
    mask_year = (years == ref_year)
    mask_jan_may = (months < 5) | ((months == 5) & (days <= 31))
    mask = mask_year & mask_jan_may

    da_sel = da.where(mask, drop=True)
    if da_sel.sizes.get("time", 0) == 0:
        print(f"[BlockB-2] WARNING: {case_name} has no Jan–May data in {ref_year}.")
        return None

    print(
        f"[BlockB-2] {case_name}: {var_name} Jan–May range = "
        f"{da_sel.time.min().values} -> {da_sel.time.max().values} "
        f"(n={da_sel.sizes['time']})"
    )
    return da_sel


def plot_group_metric_calendar(
    group_cfg: dict,
    metric_cfg: dict,
    scores_root: str = SCORES_ROOT_DIR,
    fig_dir:     str = FIG_OUT_DIR,
):
    """为一个 group + 指标 画 Jan–May 日历时间轴图。"""
    os.makedirs(fig_dir, exist_ok=True)

    gid    = group_cfg["group_id"]
    gname  = group_cfg["name"]
    gtitle = group_cfg["title"]
    cases  = group_cfg["cases"]

    var_name = metric_cfg["var_name"]
    pretty   = metric_cfg["pretty"]
    ylabel   = metric_cfg["ylabel"]
    y_lim    = Y_LIM_DEFAULTS.get(var_name, None)

    print(f"\n[BlockB-2] ==== Calendar group {gid}, metric={var_name} ====")

    series = []
    for cfg in cases:
        case_name = cfg["case_name"]
        da = load_metric_until_end_of_may(
            case_name=case_name,
            var_name=var_name,
            root_dir=scores_root,
        )
        if da is None:
            continue

        doy = da.time.dt.dayofyear.values.astype(int)
        series.append((cfg, doy, da))

    if not series:
        print(f"[BlockB-2]  No valid series for group {gid}, metric={var_name}, skip.")
        return

    fig, ax = plt.subplots(figsize=(8, 5))

    for cfg, doy, da in series:
        ax.plot(
            doy,
            da.values,
            label=cfg["label"],
            color=cfg.get("color", None),
            linewidth=2,
        )

    # no-leap: Jan1=1, Feb1=32, Mar1=60, Apr1=91, May1=121, May31=151
    month_starts_doy = [1, 32, 60, 91, 121]
    month_labels     = ["Jan", "Feb", "Mar", "Apr", "May"]

    ax.set_xlim(1, 151)
    ax.set_xticks(month_starts_doy)
    ax.set_xticklabels(month_labels, fontsize=11)

    if y_lim is not None:
        ax.set_ylim(y_lim[0], y_lim[1])

    ax.set_xlabel("Calendar time (year 0008, Jan–May)", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(
        gtitle + f"\nMetric: {pretty} (X-axis: calendar months Jan–May)",
        fontsize=13,
    )

    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)
    fig.tight_layout()

    base_name = f"{var_name}_calendar_group{gid}_{gname}_JanMay"
    out_png = os.path.join(fig_dir, base_name + ".png")
    out_pdf = os.path.join(fig_dir, base_name + ".pdf")
    fig.savefig(out_png, dpi=300)
    fig.savefig(out_pdf, dpi=300)


def main_blockB_calendar():
    print("[BlockB-2] Start calendar-time plots for all metrics ...")
    for metric_cfg in METRICS:
        var_name = metric_cfg["var_name"]
        print(f"\n[BlockB-2] === Metric: {var_name} ===")
        for g in GROUPS:
            plot_group_metric_calendar(
                group_cfg=g,
                metric_cfg=metric_cfg,
                scores_root=SCORES_ROOT_DIR,
                fig_dir=FIG_OUT_DIR,
            )
    print("[BlockB-2] Calendar-time plots done.")

# ----------------- 总入口：两套一起画 -----------------
def main_blockB():
    main_blockB_lead()
    main_blockB_calendar()

if __name__ == "__main__":
    main_blockB()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockC (Ultimate Debug) — Hindcast Polar Minimum Temperature (60–90 N)
======================================================================
Refactored to match BlockA's robust structure.
Key goal: Kill the (30, 121, 23, 23) ghost dimension issue.
"""

import os
import shutil
from pathlib import Path
import numpy as np
import xarray as xr

# ---------------------------------------------------------------------
# Paths & Config
# ---------------------------------------------------------------------
HINDCAST_BASE_DIR = "/home/weiji/restart_exam/hindcast_data"
REF_PATH = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc"
OUT_DIR = "/home/weiji/test_code/plots/hindcast/Tmin"

os.makedirs(OUT_DIR, exist_ok=True)


# ---------------------------------------------------------------------
# Scoring Functions
# ---------------------------------------------------------------------
def crps_ensemble(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    ens = np.asarray(ensemble)
    obs_arr = np.asarray(obs)
    if obs_arr.ndim == 1:
        obs_arr = obs_arr[None, :]
    diff_to_obs = np.abs(ens - obs_arr)
    term1 = diff_to_obs.mean(axis=0)
    diff_ij = np.abs(ens[:, None, ...] - ens[None, :, ...])
    term2 = 0.5 * diff_ij.mean(axis=(0, 1))
    return term1 - term2

def ensemble_spread(ensemble: np.ndarray) -> np.ndarray:
    return np.asarray(ensemble).std(axis=0, ddof=0)

def ensemble_bias(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    return np.asarray(ensemble).mean(axis=0) - np.asarray(obs)

def ensemble_mae(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    return np.abs(np.asarray(ensemble) - np.asarray(obs)).mean(axis=0)


# ---------------------------------------------------------------------
# Core Logic: Load, Clean, and Reduce ONE file
# ---------------------------------------------------------------------
def load_and_process_single_member(fpath: Path, member_id: int) -> xr.DataArray | None:
    """
    读取单个成员文件，立即清洗维度，并计算极区最低温。
    返回: DataArray (time, lev) [无 member 维，后续再 expand]
    """
    try:
        ds = xr.open_dataset(fpath, decode_times=True)
        
        # 1. 找变量
        vname = "T" if "T" in ds else "TEMP"
        if vname not in ds:
            print(f"    [WARN] No T/TEMP in {fpath.name}")
            ds.close()
            return None
        
        da = ds[vname]
        
        # [DEBUG] 打印原始维度
        if member_id == 1: # 只打印第一个成员的详细信息，避免刷屏
            print(f"    [DEBUG-File] Raw dims: {da.dims}")

        # 2. 【核心修复】处理垂直维双胞胎 (plev & plev_2)
        # 只要发现 plev_2，立刻杀掉
        if "plev_2" in da.dims:
            if "plev" in da.dims:
                # 两个都有，删掉 plev_2
                if member_id == 1: print("    [DEBUG-File] Found both plev/plev_2 -> Dropping plev_2")
                da = da.isel(plev_2=0, drop=True) 
            else:
                # 只有 plev_2，改名为 plev
                if member_id == 1: print("    [DEBUG-File] Found only plev_2 -> Renaming to plev")
                da = da.rename({"plev_2": "plev"})
        
        # 处理可能的 ilev
        if "ilev" in da.dims:
            da = da.isel(ilev=0, drop=True)

        # 3. 维度重命名 (lon/lat -> lon/lat)
        rename_map = {}
        for d in da.dims:
            if d.lower().startswith("lon"): rename_map[d] = "lon"
            elif d.lower().startswith("lat"): rename_map[d] = "lat"
            elif d in ["lev_p", "level"]: rename_map[d] = "plev"
        
        if rename_map:
            da = da.rename(rename_map)

        # 4. 立即计算极区最低温 (减少数据量，防止后续内存爆炸)
        #    Target: Tmin(60-90N)
        
        # 4a. 经向平均
        if "lon" in da.dims:
            da = da.mean(dim="lon")
        
        # 4b. 纬向最小值
        if "lat" in da.dims:
            da_cap = da.sel(lat=slice(60, 90))
            if da_cap.sizes["lat"] == 0:
                raise ValueError(f"Empty lat slice 60-90N in {fpath.name}")
            da = da_cap.min(dim="lat")

        # 5. 再次确认垂直维
        # 此时 da 应该只有 (time, plev)
        dims = list(da.dims)
        vertical_dim = None
        for d in dims:
            if d in ["plev", "lev"]:
                vertical_dim = d
                break
        
        if not vertical_dim:
            print(f"    [ERROR] No vertical dim found after processing: {dims}")
            ds.close(); return None

        # 统一改为 'lev'
        if vertical_dim != "lev":
            da = da.rename({vertical_dim: "lev"})

        # 6. 单位转换 (hPa -> Pa)
        lev_vals = da.lev.values
        if np.nanmax(lev_vals) <= 2000.0:
            da = da.assign_coords(lev=lev_vals * 100.0)
            da.lev.attrs["units"] = "Pa"

        # 7. 最终清洗 (Squeeze & Drop Coords)
        da = da.squeeze()
        da = da.reset_coords(drop=True)
        
        # [DEBUG] 检查清洗后的结果
        if member_id == 1:
            print(f"    [DEBUG-File] Cleaned dims: {da.dims}, Shape: {da.shape}")
            if len(da.shape) != 2:
                print(f"    [FATAL] Expected 2 dims (time, lev), got {da.shape}!")

        ds.close()
        
        # 加上 member 维
        da = da.expand_dims("member")
        da = da.assign_coords(member=[member_id])
        
        return da

    except Exception as e:
        print(f"    [ERROR] Reading {fpath.name}: {e}")
        return None


# ---------------------------------------------------------------------
# Loader: Load full ensemble for a case
# ---------------------------------------------------------------------
def load_case_ensemble(case_dir: Path) -> xr.DataArray | None:
    """读取一个 case 下所有的成员，并合并返回 (member, time, lev)"""
    t_dir = case_dir / "T"
    if not t_dir.is_dir():
        return None
    
    files = sorted(t_dir.glob("*.nc"))
    if not files:
        return None

    print(f"\n[LOADER] Loading Case: {case_dir.name} ({len(files)} files)")

    da_list = []
    for idx, f in enumerate(files):
        # idx+1 作为 member id
        da = load_and_process_single_member(f, idx + 1)
        if da is not None:
            da_list.append(da)

    if not da_list:
        print("    [WARN] No valid data found.")
        return None

    # 合并
    try:
        # 此时 list 里的每个 da 都是 (1, time, lev)
        ens = xr.concat(da_list, dim="member")
        print(f"    [LOADER] Concatenated shape: {ens.shape} (member, time, lev)")
        
        if ens.ndim != 3:
            raise ValueError(f"Concatenation resulted in {ens.ndim} dims! Expected 3.")
            
        return ens
    except Exception as e:
        print(f"    [ERROR] Concat failed: {e}")
        return None


# ---------------------------------------------------------------------
# Reference Loader
# ---------------------------------------------------------------------
def load_reference(ref_path: str):
    print(f"\n[REF] Loading Reference: {ref_path}")
    # 复用那个单文件处理逻辑，传入 ID=0 仅作标记
    da = load_and_process_single_member(Path(ref_path), member_id=1)
    if da is None:
        raise RuntimeError("Failed to load reference file!")
    
    # Reference 不需要 member 维
    da = da.isel(member=0, drop=True)
    print(f"    [REF] Final shape: {da.shape} (time, lev)")
    return da


# ---------------------------------------------------------------------
# Compute Scores
# ---------------------------------------------------------------------
def compute_scores_for_case(case_name, ens_da, ref_da):
    """
    ens_da: (member, time, lev)
    ref_da: (time, lev)
    """
    # Align time and lev
    print(f"  [SCORE] Aligning... Ens time={ens_da.time.size}, Ref time={ref_da.time.size}")
    ens_da, ref_da = xr.align(ens_da, ref_da, join="inner")
    
    if ens_da.time.size == 0:
        print("  [ERROR] No overlapping time points!")
        return None

    ens = ens_da.values
    obs = ref_da.values
    
    # 终极检查
    print(f"  [SCORE] Numpy shapes -> Ens: {ens.shape}, Obs: {obs.shape}")
    if ens.ndim != 3:
        print(f"  [FATAL] Ens has {ens.ndim} dims. Cannot compute scores.")
        return None

    n_member, n_time, n_lev = ens.shape
    
    CRPS = np.zeros((n_time, n_lev))
    Spread = np.zeros_like(CRPS)
    Bias = np.zeros_like(CRPS)
    MAE = np.zeros_like(CRPS)

    for k in range(n_lev):
        ens_k = ens[:, :, k]
        obs_k = obs[:, k]
        CRPS[:, k]   = crps_ensemble(ens_k, obs_k)
        Spread[:, k] = ensemble_spread(ens_k)
        Bias[:, k]   = ensemble_bias(ens_k, obs_k)
        MAE[:, k]    = ensemble_mae(ens_k, obs_k)

    # 封装
    ds_out = xr.Dataset(
        coords={"time": ens_da.time, "lev": ens_da.lev},
        data_vars={
            "CRPS": (("time", "lev"), CRPS),
            "spread": (("time", "lev"), Spread),
            "bias": (("time", "lev"), Bias),
            "mae": (("time", "lev"), MAE),
        },
        attrs={"case_name": case_name, "units": "K"}
    )
    return ds_out


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main_blockC():
    print("[BlockC] Starting Hindcast Tmin Verification...")
    
    # 1. Load Reference
    try:
        ref_da = load_reference(REF_PATH)
    except Exception as e:
        print(e); return

    # 2. Find cases
    case_dirs = sorted(list(Path(HINDCAST_BASE_DIR).glob("0008-*")))
    
    for case_dir in case_dirs:
        case_name = case_dir.name
        out_file = os.path.join(OUT_DIR, f"Tmin_scores_{case_name}.nc")
        
        # Cleanup old file
        if os.path.exists(out_file):
            os.remove(out_file)

        # 3. Load & Process Ensemble
        ens_da = load_case_ensemble(case_dir)
        if ens_da is None:
            continue
            
        # 4. Compute Scores
        ds_scores = compute_scores_for_case(case_name, ens_da, ref_da)
        
        if ds_scores is not None:
            ds_scores.to_netcdf(out_file)
            print(f"  [SUCCESS] Saved: {out_file}")
            
    print("\n[BlockC] All Done.")

if __name__ == "__main__":
    main_blockC()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockD (Final Complete) — Visualization for Polar Minimum Temperature (Tmin)
==========================================================================
Features:
  1. Auto-detects Calendar (Leap vs No-Leap) from data attributes.
  2. Plots Lead-time lines (50 & 100 hPa).
  3. Plots Calendar-time lines (50 & 100 hPa).
  4. Plots 2D Time-Pressure sections (Calendar x Pressure).
"""

import os
import re
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ----------------- 基础配置 -----------------
SCORES_ROOT_DIR = "/home/weiji/test_code/plots/hindcast/Tmin"
FIG_OUT_DIR     = SCORES_ROOT_DIR
os.makedirs(FIG_OUT_DIR, exist_ok=True)

# 关注的层 (hPa)
TARGET_LEVELS = [50, 100]

# 绘图时间范围 (Lead Days)
LEAD_START = 1
LEAD_END   = 150

# 指标与配色
METRICS = [
    {"var": "CRPS",   "pretty": "CRPS",            "ylabel": "CRPS [K]",        "cmap": "OrRd",   "min_zero": True},
    {"var": "bias",   "pretty": "Mean Error",      "ylabel": "Mean Error [K]",  "cmap": "RdBu_r", "min_zero": False},
    {"var": "spread", "pretty": "Ensemble Spread", "ylabel": "Spread [K]",      "cmap": "YlGnBu", "min_zero": True},
]

# ----------------- 分组配置 (复用 BlockB) -----------------
GROUPS = [
    {
        "group_id": 0, "name": "group0_small_init", "title": "Group 0 – Small perturbation, different init month",
        "cases": [
            {"case_name": "0008-01", "label": "Jan small (0008-01)", "color": "C0"},
            {"case_name": "0008-02", "label": "Feb small (0008-02)", "color": "C1"},
            {"case_name": "0008-03", "label": "Mar small (0008-03)", "color": "C2"},
        ]
    },
    {
        "group_id": 1, "name": "group1_Feb_chem_vs_no", "title": "Group 1 – Feb: Chem vs Prescribe-O3",
        "cases": [
            {"case_name": "0008-02",         "label": "Feb Chem (0008-02)",         "color": "C0"},
            {"case_name": "0008-02_NOCOUPL", "label": "Feb Prescribe-O3 (NOCOUPL)", "color": "C3"},
        ]
    },
    {
        "group_id": 2, "name": "group2_Mar_chem_vs_no", "title": "Group 2 – Mar: Chem vs Prescribe-O3",
        "cases": [
            {"case_name": "0008-03",         "label": "Mar Chem (0008-03)",         "color": "C0"},
            {"case_name": "0008-03_NOCOUPL", "label": "Mar Prescribe-O3 (NOCOUPL)", "color": "C3"},
        ]
    },
    {
        "group_id": 3, "name": "group3_largepert", "title": "Group 3 – Large perturbation, different init month",
        "cases": [
            {"case_name": "0008-02_v2", "label": "Feb large (0008-02_v2)", "color": "C0"},
            {"case_name": "0008-03_v2", "label": "Mar large (0008-03_v2)", "color": "C1"},
            {"case_name": "0008-04_v2", "label": "Apr large (0008-04_v2)", "color": "C2"},
        ]
    },
    {
        "group_id": 4, "name": "group4_Mar_diff_pert", "title": "Group 4 – Mar Init: Different perturbations",
        "cases": [
            {"case_name": "0008-03",    "label": "Small-pert (0008-03)",    "color": "C0"},
            {"case_name": "0008-03_v2", "label": "Large-pert (0008-03_v2)", "color": "C1"},
            {"case_name": "0008-03_v3", "label": "Q-pert (0008-03_v3)",     "color": "C2"},
        ]
    },
    {
        "group_id": 5, "name": "group5_Qpert_diff_init", "title": "Group 5 – Q-pert, different init month",
        "cases": [
            {"case_name": "0008-02_v3", "label": "Feb Q-pert (0008-02_v3)", "color": "C0"},
            {"case_name": "0008-03_v3", "label": "Mar Q-pert (0008-03_v3)", "color": "C1"},
        ]
    },
    {
        "group_id": 6, "name": "group6_Feb_diff_pert", "title": "Group 6 – Feb Init: Different perturbations",
        "cases": [
            {"case_name": "0008-02",    "label": "Small-pert (0008-02)",    "color": "C0"},
            {"case_name": "0008-02_v2", "label": "Large-pert (0008-02_v2)", "color": "C1"},
            {"case_name": "0008-02_v3", "label": "Q-pert (0008-02_v3)",     "color": "C2"},
        ]
    },
]

# ----------------- 核心工具：智能日历检测 -----------------

def load_case_ds(case_name):
    nc_path = os.path.join(SCORES_ROOT_DIR, f"Tmin_scores_{case_name}.nc")
    if not os.path.exists(nc_path): return None
    try:
        ds = xr.open_dataset(nc_path)
        return ds
    except Exception as e:
        print(f"[Error] Open failed {case_name}: {e}")
        return None

def determine_anchor_year(ds, case_name):
    """
    检查数据中的 'calendar' 属性，决定绘图时使用的锚点年份。
    - 'noleap' / '365_day': 返回 2001 (平年)
    - 'gregorian' / 'standard' / 'all_leap': 返回 2000 (闰年，适配 0008)
    """
    # 1. 尝试从 time 属性获取日历信息
    cal = ds.time.encoding.get('calendar', '') or ds.time.attrs.get('calendar', '')
    cal = str(cal).lower()
    
    anchor_year = 2001 # 默认为平年 (Safe bet for WACCM)
    cal_type = "Unknown (Assume NoLeap)"

    if 'noleap' in cal or '365_day' in cal:
        anchor_year = 2001
        cal_type = "NoLeap"
    elif 'gregorian' in cal or 'standard' in cal or 'proleptic' in cal:
        # 0008 是闰年，所以如果模式是标准历，我们需要一个闰年作为锚点
        anchor_year = 2000 
        cal_type = "Gregorian/Leap"
    elif 'all_leap' in cal or '366_day' in cal:
        anchor_year = 2000
        cal_type = "AllLeap"
    
    # print(f"    [Calendar Check] Case: {case_name} | Attr: '{cal}' -> Using Year {anchor_year} ({cal_type})")
    return anchor_year

def reconstruct_dates(ds, case_name):
    """
    将 timedelta/int 时间轴重建为 DateTime 对象，用于绘图。
    """
    # 1. 解析起始月份
    match = re.search(r"(\d{4})-(\d{2})", case_name)
    start_month = int(match.group(2)) if match else 1
    
    # 2. 决定年份 (平年 vs 闰年)
    anchor_year = determine_anchor_year(ds, case_name)
    
    # 3. 构造起始日期
    start_date = pd.Timestamp(f"{anchor_year}-{start_month:02d}-01")
    
    # 4. 生成时间序列
    n_steps = ds.sizes["time"]
    # 假设日数据 (freq='D')
    dates = pd.date_range(start=start_date, periods=n_steps, freq='D')
    
    return dates

# ============================================================
# 1. Lead-time 折线图
# ============================================================

def plot_group_lead_lines(group_cfg, metric_cfg, level_hpa, fig_dir=FIG_OUT_DIR):
    var_name = metric_cfg["var"]
    target_pa = level_hpa * 100.0
    
    fig, ax = plt.subplots(figsize=(8, 5))
    has_data = False

    for cfg in group_cfg["cases"]:
        ds = load_case_ds(cfg["case_name"])
        if ds is None or var_name not in ds: continue
        try:
            da = ds[var_name]
            da = da.sel(lev=target_pa, method="nearest", tolerance=100)
            
            # 截取
            n_time = da.sizes["time"]
            eff_end = min(LEAD_END, n_time)
            idx_start = max(0, LEAD_START - 1)
            
            # X轴: Lead Days (整数)
            lead_days = np.arange(idx_start, eff_end)
            data_vals = da.isel(time=slice(idx_start, eff_end)).values
            
            ax.plot(lead_days, data_vals, label=cfg["label"], color=cfg.get("color"), lw=2, alpha=0.8)
            has_data = True
        except: pass
        finally: ds.close()

    if not has_data: plt.close(fig); return

    ax.set_xlabel("Lead Time (Days)", fontsize=12)
    ax.set_ylabel(metric_cfg["ylabel"], fontsize=12)
    ax.set_title(f"{group_cfg['title']}\n{metric_cfg['pretty']} @ {level_hpa} hPa (Lead Time)", fontweight='bold')
    ax.set_xlim(LEAD_START, LEAD_END)
    ax.grid(True, alpha=0.3)
    ax.legend()
    if metric_cfg["min_zero"]: ax.set_ylim(0, ax.get_ylim()[1])
    
    fname = f"{var_name}_lead_{level_hpa}hPa_group{group_cfg['group_id']}.png"
    fig.savefig(os.path.join(fig_dir, fname), dpi=300)
    plt.close(fig)
    print(f"  -> Lead Plot: {fname}")

# ============================================================
# 2. Calendar-time 折线图 (Jan-May)
# ============================================================

def plot_group_calendar_lines(group_cfg, metric_cfg, level_hpa, fig_dir=FIG_OUT_DIR):
    var_name = metric_cfg["var"]
    target_pa = level_hpa * 100.0
    
    fig, ax = plt.subplots(figsize=(8, 5))
    has_data = False

    for cfg in group_cfg["cases"]:
        ds = load_case_ds(cfg["case_name"])
        if ds is None or var_name not in ds: continue
        try:
            da = ds[var_name].sel(lev=target_pa, method="nearest", tolerance=100)
            
            # 重建日期
            dates = reconstruct_dates(ds, cfg["case_name"])
            
            # 筛选 Jan-May
            # 这里的 dates 已经是带年份的 Timestamp，我们可以利用 month 属性
            mask = (dates.month >= 1) & (dates.month <= 5)
            
            if np.any(mask):
                plot_dates = dates[mask]
                plot_vals  = da.values[mask]
                
                # 绘图
                ax.plot(plot_dates, plot_vals, label=cfg["label"], color=cfg.get("color"), lw=2, alpha=0.8)
                has_data = True
        except Exception as e:
            # print(f"Err {cfg['case_name']}: {e}")
            pass
        finally: ds.close()

    if not has_data: plt.close(fig); return

    # 格式化 X 轴
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    
    ax.set_xlabel("Calendar Time", fontsize=12)
    ax.set_ylabel(metric_cfg["ylabel"], fontsize=12)
    ax.set_title(f"{group_cfg['title']}\n{metric_cfg['pretty']} @ {level_hpa} hPa (Calendar)", fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    if metric_cfg["min_zero"]: ax.set_ylim(0, ax.get_ylim()[1])

    fname = f"{var_name}_cal_{level_hpa}hPa_group{group_cfg['group_id']}.png"
    fig.savefig(os.path.join(fig_dir, fname), dpi=300)
    plt.close(fig)
    print(f"  -> Calendar Plot: {fname}")

# ============================================================
# 3. Time-Pressure 2D 填色图
# ============================================================

def plot_2d_section(case_name, metric_cfg, fig_dir=FIG_OUT_DIR):
    ds = load_case_ds(case_name)
    if ds is None: return

    var_name = metric_cfg["var"]
    if var_name not in ds: ds.close(); return

    try:
        # 1. 准备数据
        Z = ds[var_name].transpose("lev", "time").values
        Y = ds.lev.values
        if np.max(Y) > 2000: Y = Y / 100.0 # hPa
        
        # 2. 准备 X 轴 (日期)
        dates = reconstruct_dates(ds, case_name)
        
        # 3. 绘图
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Color Range
        if var_name == "bias":
            vmax = np.nanmax(np.abs(Z)) or 1
            vmin = -vmax
        else:
            vmax = np.nanmax(Z) or 1
            vmin = 0
            
        pcm = ax.pcolormesh(dates, Y, Z, shading="nearest", cmap=metric_cfg["cmap"], vmin=vmin, vmax=vmax)
        
        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(1000, 1)
        ax.set_ylabel("Pressure (hPa)", fontsize=11)
        
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        
        ax.set_title(f"Case: {case_name} — {metric_cfg['pretty']} (Tmin)", fontweight='bold')
        plt.colorbar(pcm, ax=ax, label=metric_cfg["ylabel"])
        
        fname = f"{var_name}_TimePress_{case_name}.png"
        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, fname), dpi=300)
        print(f"  -> 2D Plot: {fname}")
        
    except Exception as e:
        print(f"  [Error 2D] {case_name}: {e}")
    finally:
        ds.close(); plt.close(fig)

# ============================================================
# Main Entry
# ============================================================

def main_blockD():
    print(f"[BlockD] Output: {FIG_OUT_DIR}")
    
    # 1. Lead-time Plots
    print("\n--- Generating Lead-time Plots ---")
    for m in METRICS:
        for g in GROUPS:
            for lev in TARGET_LEVELS:
                plot_group_lead_lines(g, m, lev)
    
    # 2. Calendar-time Plots
    print("\n--- Generating Calendar-time Plots ---")
    for m in METRICS:
        for g in GROUPS:
            for lev in TARGET_LEVELS:
                plot_group_calendar_lines(g, m, lev)
    
    # 3. 2D Plots (Find all cases)
    print("\n--- Generating 2D Time-Pressure Plots ---")
    all_cases = set()
    for g in GROUPS:
        for c in g["cases"]:
            all_cases.add(c["case_name"])
            
    for cname in sorted(list(all_cases)):
        for m in METRICS:
            plot_2d_section(cname, m)

    print("\n[BlockD] All Finished.")

if __name__ == "__main__":
    main_blockD()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockE — Hindcast Zonal Mean Zonal Wind at 60N (U60N)
=====================================================
Structure matches BlockC (Ultimate Debug).
Target: U wind, Zonal Mean, interpolated to 60°N.
"""

import os
import shutil
from pathlib import Path
import numpy as np
import xarray as xr

# ---------------------------------------------------------------------
# Paths & Config
# ---------------------------------------------------------------------
HINDCAST_BASE_DIR = "/home/weiji/restart_exam/hindcast_data"
REF_PATH = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc"
# 输出目录改为 U
OUT_DIR = "/home/weiji/test_code/plots/hindcast/U"

os.makedirs(OUT_DIR, exist_ok=True)


# ---------------------------------------------------------------------
# Scoring Functions (Same as before)
# ---------------------------------------------------------------------
def crps_ensemble(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    ens = np.asarray(ensemble)
    obs_arr = np.asarray(obs)
    if obs_arr.ndim == 1:
        obs_arr = obs_arr[None, :]
    diff_to_obs = np.abs(ens - obs_arr)
    term1 = diff_to_obs.mean(axis=0)
    diff_ij = np.abs(ens[:, None, ...] - ens[None, :, ...])
    term2 = 0.5 * diff_ij.mean(axis=(0, 1))
    return term1 - term2

def ensemble_spread(ensemble: np.ndarray) -> np.ndarray:
    return np.asarray(ensemble).std(axis=0, ddof=0)

def ensemble_bias(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    return np.asarray(ensemble).mean(axis=0) - np.asarray(obs)

def ensemble_mae(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    return np.abs(np.asarray(ensemble) - np.asarray(obs)).mean(axis=0)


# ---------------------------------------------------------------------
# Core Logic: Load, Clean, and Reduce ONE file
# ---------------------------------------------------------------------
def load_and_process_single_member(fpath: Path, member_id: int) -> xr.DataArray | None:
    """
    读取单个成员，清洗维度，计算 U @ 60N。
    """
    try:
        ds = xr.open_dataset(fpath, decode_times=True)
        
        # 1. 找变量 U
        if "U" not in ds:
            print(f"    [WARN] No variable 'U' in {fpath.name}")
            ds.close()
            return None
        
        da = ds["U"]
        
        # [DEBUG] 打印第一个成员的维度
        if member_id == 1:
            print(f"    [DEBUG-File] Raw dims: {da.dims}")

        # 2. 【核心修复】杀掉 plev_2
        if "plev_2" in da.dims:
            if "plev" in da.dims:
                if member_id == 1: print("    [DEBUG-File] Found both plev/plev_2 -> Dropping plev_2")
                da = da.isel(plev_2=0, drop=True) 
            else:
                if member_id == 1: print("    [DEBUG-File] Found only plev_2 -> Renaming to plev")
                da = da.rename({"plev_2": "plev"})
        
        if "ilev" in da.dims:
            da = da.isel(ilev=0, drop=True)

        # 3. 维度重命名
        rename_map = {}
        for d in da.dims:
            if d.lower().startswith("lon"): rename_map[d] = "lon"
            elif d.lower().startswith("lat"): rename_map[d] = "lat"
            elif d in ["lev_p", "level"]: rename_map[d] = "plev"
        
        if rename_map:
            da = da.rename(rename_map)

        # 4. 计算 U @ 60N (Zonal Mean + Lat Selection)
        
        # 4a. 经向平均 (Zonal Mean)
        if "lon" in da.dims:
            da = da.mean(dim="lon")
        
        # 4b. 选择 60N
        if "lat" in da.dims:
            # 使用 nearest 选择最接近 60N 的纬度
            da = da.sel(lat=60.0, method="nearest")
            # 此时 lat 维度还在（但在 xarray 中变成了 scalar coord），或者是没了
            # 无论如何，数据维度应该只剩下 (time, plev)

        # 5. 确认垂直维
        dims = list(da.dims)
        vertical_dim = None
        for d in dims:
            if d in ["plev", "lev"]:
                vertical_dim = d
                break
        
        if not vertical_dim:
            print(f"    [ERROR] No vertical dim found: {dims}")
            ds.close(); return None

        if vertical_dim != "lev":
            da = da.rename({vertical_dim: "lev"})

        # 6. 单位转换 (hPa -> Pa 统一存储)
        lev_vals = da.lev.values
        if np.nanmax(lev_vals) <= 2000.0:
            da = da.assign_coords(lev=lev_vals * 100.0)
            da.lev.attrs["units"] = "Pa"

        # 7. 最终清洗
        da = da.squeeze()
        da = da.reset_coords(drop=True)
        
        if member_id == 1:
            print(f"    [DEBUG-File] Cleaned dims: {da.dims}, Shape: {da.shape}")

        ds.close()
        
        da = da.expand_dims("member")
        da = da.assign_coords(member=[member_id])
        
        return da

    except Exception as e:
        print(f"    [ERROR] Reading {fpath.name}: {e}")
        return None


# ---------------------------------------------------------------------
# Loaders (Same as BlockC)
# ---------------------------------------------------------------------
def load_case_ensemble(case_dir: Path) -> xr.DataArray | None:
    # 注意这里目录是 /U/ 而不是 /T/
    u_dir = case_dir / "U"
    if not u_dir.is_dir():
        # 有时候可能在大写 U 或者小写 u，或者 Uwind，根据你之前的描述是 "U"
        return None
    
    files = sorted(u_dir.glob("*.nc"))
    if not files:
        return None

    print(f"\n[LOADER] Loading Case: {case_dir.name} ({len(files)} files)")

    da_list = []
    for idx, f in enumerate(files):
        da = load_and_process_single_member(f, idx + 1)
        if da is not None:
            da_list.append(da)

    if not da_list:
        print("    [WARN] No valid data found.")
        return None

    try:
        ens = xr.concat(da_list, dim="member")
        print(f"    [LOADER] Concatenated shape: {ens.shape}")
        if ens.ndim != 3:
            raise ValueError(f"Concatenation resulted in {ens.ndim} dims! Expected 3.")
        return ens
    except Exception as e:
        print(f"    [ERROR] Concat failed: {e}")
        return None

def load_reference(ref_path: str):
    print(f"\n[REF] Loading Reference: {ref_path}")
    da = load_and_process_single_member(Path(ref_path), member_id=1)
    if da is None:
        raise RuntimeError("Failed to load reference file!")
    da = da.isel(member=0, drop=True)
    print(f"    [REF] Final shape: {da.shape}")
    return da

# ---------------------------------------------------------------------
# Compute Scores
# ---------------------------------------------------------------------
def compute_scores_for_case(case_name, ens_da, ref_da):
    print(f"  [SCORE] Aligning... Ens time={ens_da.time.size}, Ref time={ref_da.time.size}")
    ens_da, ref_da = xr.align(ens_da, ref_da, join="inner")
    
    if ens_da.time.size == 0:
        print("  [ERROR] No overlapping time points!")
        return None

    ens = ens_da.values
    obs = ref_da.values
    
    if ens.ndim != 3:
        print(f"  [FATAL] Ens has {ens.ndim} dims. Cannot compute scores.")
        return None

    n_member, n_time, n_lev = ens.shape
    
    CRPS = np.zeros((n_time, n_lev))
    Spread = np.zeros_like(CRPS)
    Bias = np.zeros_like(CRPS)
    MAE = np.zeros_like(CRPS)

    for k in range(n_lev):
        ens_k = ens[:, :, k]
        obs_k = obs[:, k]
        CRPS[:, k]   = crps_ensemble(ens_k, obs_k)
        Spread[:, k] = ensemble_spread(ens_k)
        Bias[:, k]   = ensemble_bias(ens_k, obs_k)
        MAE[:, k]    = ensemble_mae(ens_k, obs_k)

    ds_out = xr.Dataset(
        coords={"time": ens_da.time, "lev": ens_da.lev},
        data_vars={
            "CRPS": (("time", "lev"), CRPS),
            "spread": (("time", "lev"), Spread),
            "bias": (("time", "lev"), Bias),
            "mae": (("time", "lev"), MAE),
        },
        attrs={
            "case_name": case_name, 
            "units": "m/s",
            "description": "Zonal Mean Zonal Wind at 60N"
        }
    )
    return ds_out


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main_blockE():
    print("[BlockE] Starting Hindcast U (60N) Verification...")
    
    try:
        ref_da = load_reference(REF_PATH)
    except Exception as e:
        print(e); return

    case_dirs = sorted(list(Path(HINDCAST_BASE_DIR).glob("0008-*")))
    
    for case_dir in case_dirs:
        case_name = case_dir.name
        # 文件名前缀改为 U_scores_
        out_file = os.path.join(OUT_DIR, f"U_scores_{case_name}.nc")
        
        if os.path.exists(out_file):
            os.remove(out_file)

        ens_da = load_case_ensemble(case_dir)
        if ens_da is None:
            continue
            
        ds_scores = compute_scores_for_case(case_name, ens_da, ref_da)
        
        if ds_scores is not None:
            ds_scores.to_netcdf(out_file)
            print(f"  [SUCCESS] Saved: {out_file}")
            
    print("\n[BlockE] All Done.")

if __name__ == "__main__":
    main_blockE()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockF — Visualization for Zonal Mean U at 60N
==============================================
Based on BlockD (Final Complete).
Features:
  1. Target Levels: 10, 50, 100 hPa.
  2. Metrics: U wind scores (m/s).
  3. Includes Lead-time, Calendar-time, and 2D plots.
"""

import os
import re
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ----------------- 基础配置 -----------------
# 路径改为 U
SCORES_ROOT_DIR = "/home/weiji/test_code/plots/hindcast/U"
FIG_OUT_DIR     = SCORES_ROOT_DIR
os.makedirs(FIG_OUT_DIR, exist_ok=True)

# 【核心修改】增加 10 hPa
TARGET_LEVELS = [10, 50, 100]

LEAD_START = 1
LEAD_END   = 150

# 指标配置 (单位 m/s)
METRICS = [
    {"var": "CRPS",   "pretty": "CRPS",            "ylabel": "CRPS [m/s]",       "cmap": "OrRd",   "min_zero": True},
    {"var": "bias",   "pretty": "Mean Error",      "ylabel": "Mean Error [m/s]", "cmap": "RdBu_r", "min_zero": False},
    {"var": "spread", "pretty": "Ensemble Spread", "ylabel": "Spread [m/s]",     "cmap": "YlGnBu", "min_zero": True},
]

# ----------------- 分组配置 (不变) -----------------
GROUPS = [
    {
        "group_id": 0, "name": "group0_small_init", "title": "Group 0 – Small perturbation",
        "cases": [
            {"case_name": "0008-01", "label": "Jan small (0008-01)", "color": "C0"},
            {"case_name": "0008-02", "label": "Feb small (0008-02)", "color": "C1"},
            {"case_name": "0008-03", "label": "Mar small (0008-03)", "color": "C2"},
        ]
    },
    {
        "group_id": 1, "name": "group1_Feb_chem_vs_no", "title": "Group 1 – Feb: Chem vs Prescribe",
        "cases": [
            {"case_name": "0008-02",         "label": "Feb Chem",      "color": "C0"},
            {"case_name": "0008-02_NOCOUPL", "label": "Feb Prescribe", "color": "C3"},
        ]
    },
    {
        "group_id": 2, "name": "group2_Mar_chem_vs_no", "title": "Group 2 – Mar: Chem vs Prescribe",
        "cases": [
            {"case_name": "0008-03",         "label": "Mar Chem",      "color": "C0"},
            {"case_name": "0008-03_NOCOUPL", "label": "Mar Prescribe", "color": "C3"},
        ]
    },
    {
        "group_id": 3, "name": "group3_largepert", "title": "Group 3 – Large perturbation",
        "cases": [
            {"case_name": "0008-02_v2", "label": "Feb large", "color": "C0"},
            {"case_name": "0008-03_v2", "label": "Mar large", "color": "C1"},
            {"case_name": "0008-04_v2", "label": "Apr large", "color": "C2"},
        ]
    },
    {
        "group_id": 4, "name": "group4_Mar_diff_pert", "title": "Group 4 – Mar Init: Diff Pert",
        "cases": [
            {"case_name": "0008-03",    "label": "Small-pert", "color": "C0"},
            {"case_name": "0008-03_v2", "label": "Large-pert", "color": "C1"},
            {"case_name": "0008-03_v3", "label": "Q-pert",     "color": "C2"},
        ]
    },
    {
        "group_id": 5, "name": "group5_Qpert_diff_init", "title": "Group 5 – Q-pert",
        "cases": [
            {"case_name": "0008-02_v3", "label": "Feb Q-pert", "color": "C0"},
            {"case_name": "0008-03_v3", "label": "Mar Q-pert", "color": "C1"},
        ]
    },
    {
        "group_id": 6, "name": "group6_Feb_diff_pert", "title": "Group 6 – Feb Init: Diff Pert",
        "cases": [
            {"case_name": "0008-02",    "label": "Small-pert", "color": "C0"},
            {"case_name": "0008-02_v2", "label": "Large-pert", "color": "C1"},
            {"case_name": "0008-02_v3", "label": "Q-pert",     "color": "C2"},
        ]
    },
]

# ----------------- 核心工具 (Calendar Detection) -----------------

def load_case_ds(case_name):
    # 注意文件名是 U_scores_
    nc_path = os.path.join(SCORES_ROOT_DIR, f"U_scores_{case_name}.nc")
    if not os.path.exists(nc_path): return None
    try:
        ds = xr.open_dataset(nc_path)
        return ds
    except Exception as e:
        print(f"[Error] Open failed {case_name}: {e}")
        return None

def determine_anchor_year(ds, case_name):
    cal = ds.time.encoding.get('calendar', '') or ds.time.attrs.get('calendar', '')
    cal = str(cal).lower()
    anchor_year = 2001 
    if 'noleap' in cal or '365_day' in cal:
        anchor_year = 2001
    elif 'gregorian' in cal or 'standard' in cal or 'proleptic' in cal:
        anchor_year = 2000 
    elif 'all_leap' in cal or '366_day' in cal:
        anchor_year = 2000
    return anchor_year

def reconstruct_dates(ds, case_name):
    match = re.search(r"(\d{4})-(\d{2})", case_name)
    start_month = int(match.group(2)) if match else 1
    anchor_year = determine_anchor_year(ds, case_name)
    start_date = pd.Timestamp(f"{anchor_year}-{start_month:02d}-01")
    n_steps = ds.sizes["time"]
    dates = pd.date_range(start=start_date, periods=n_steps, freq='D')
    return dates

# ============================================================
# 1. Lead-time 折线图
# ============================================================

def plot_group_lead_lines(group_cfg, metric_cfg, level_hpa, fig_dir=FIG_OUT_DIR):
    var_name = metric_cfg["var"]
    target_pa = level_hpa * 100.0
    
    fig, ax = plt.subplots(figsize=(8, 5))
    has_data = False

    for cfg in group_cfg["cases"]:
        ds = load_case_ds(cfg["case_name"])
        if ds is None or var_name not in ds: continue
        try:
            da = ds[var_name]
            da = da.sel(lev=target_pa, method="nearest", tolerance=100)
            
            n_time = da.sizes["time"]
            eff_end = min(LEAD_END, n_time)
            idx_start = max(0, LEAD_START - 1)
            
            lead_days = np.arange(idx_start, eff_end)
            data_vals = da.isel(time=slice(idx_start, eff_end)).values
            
            ax.plot(lead_days, data_vals, label=cfg["label"], color=cfg.get("color"), lw=2, alpha=0.8)
            has_data = True
        except: pass
        finally: ds.close()

    if not has_data: plt.close(fig); return

    ax.set_xlabel("Lead Time (Days)", fontsize=12)
    ax.set_ylabel(metric_cfg["ylabel"], fontsize=12)
    ax.set_title(f"{group_cfg['title']}\n{metric_cfg['pretty']} @ {level_hpa} hPa (U60N)", fontweight='bold')
    ax.set_xlim(LEAD_START, LEAD_END)
    ax.grid(True, alpha=0.3)
    ax.legend()
    if metric_cfg["min_zero"]: ax.set_ylim(0, ax.get_ylim()[1])
    
    fname = f"{var_name}_lead_{level_hpa}hPa_group{group_cfg['group_id']}.png"
    fig.savefig(os.path.join(fig_dir, fname), dpi=300)
    plt.close(fig)
    print(f"  -> Lead Plot: {fname}")

# ============================================================
# 2. Calendar-time 折线图
# ============================================================

def plot_group_calendar_lines(group_cfg, metric_cfg, level_hpa, fig_dir=FIG_OUT_DIR):
    var_name = metric_cfg["var"]
    target_pa = level_hpa * 100.0
    
    fig, ax = plt.subplots(figsize=(8, 5))
    has_data = False

    for cfg in group_cfg["cases"]:
        ds = load_case_ds(cfg["case_name"])
        if ds is None or var_name not in ds: continue
        try:
            da = ds[var_name].sel(lev=target_pa, method="nearest", tolerance=100)
            dates = reconstruct_dates(ds, cfg["case_name"])
            mask = (dates.month >= 1) & (dates.month <= 5)
            
            if np.any(mask):
                plot_dates = dates[mask]
                plot_vals  = da.values[mask]
                ax.plot(plot_dates, plot_vals, label=cfg["label"], color=cfg.get("color"), lw=2, alpha=0.8)
                has_data = True
        except: pass
        finally: ds.close()

    if not has_data: plt.close(fig); return

    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_xlabel("Calendar Time", fontsize=12)
    ax.set_ylabel(metric_cfg["ylabel"], fontsize=12)
    ax.set_title(f"{group_cfg['title']}\n{metric_cfg['pretty']} @ {level_hpa} hPa (Calendar U60N)", fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    if metric_cfg["min_zero"]: ax.set_ylim(0, ax.get_ylim()[1])

    fname = f"{var_name}_cal_{level_hpa}hPa_group{group_cfg['group_id']}.png"
    fig.savefig(os.path.join(fig_dir, fname), dpi=300)
    plt.close(fig)
    print(f"  -> Calendar Plot: {fname}")

# ============================================================
# 3. Time-Pressure 2D
# ============================================================

def plot_2d_section(case_name, metric_cfg, fig_dir=FIG_OUT_DIR):
    ds = load_case_ds(case_name)
    if ds is None: return

    var_name = metric_cfg["var"]
    if var_name not in ds: ds.close(); return

    try:
        Z = ds[var_name].transpose("lev", "time").values
        Y = ds.lev.values
        if np.max(Y) > 2000: Y = Y / 100.0 
        
        dates = reconstruct_dates(ds, case_name)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        if var_name == "bias":
            vmax = np.nanmax(np.abs(Z)) or 1
            vmin = -vmax
        else:
            vmax = np.nanmax(Z) or 1
            vmin = 0
            
        pcm = ax.pcolormesh(dates, Y, Z, shading="nearest", cmap=metric_cfg["cmap"], vmin=vmin, vmax=vmax)
        
        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(1000, 1)
        ax.set_ylabel("Pressure (hPa)", fontsize=11)
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        
        ax.set_title(f"Case: {case_name} — {metric_cfg['pretty']} (U60N)", fontweight='bold')
        plt.colorbar(pcm, ax=ax, label=metric_cfg["ylabel"])
        
        fname = f"{var_name}_TimePress_{case_name}.png"
        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, fname), dpi=300)
        print(f"  -> 2D Plot: {fname}")
        
    except Exception as e:
        print(f"  [Error 2D] {case_name}: {e}")
    finally:
        ds.close(); plt.close(fig)

# ============================================================
# Main
# ============================================================

def main_blockF():
    print(f"[BlockF] Output: {FIG_OUT_DIR}")
    
    print("\n--- Generating Lead-time Plots ---")
    for m in METRICS:
        for g in GROUPS:
            for lev in TARGET_LEVELS:
                plot_group_lead_lines(g, m, lev)
    
    print("\n--- Generating Calendar-time Plots ---")
    for m in METRICS:
        for g in GROUPS:
            for lev in TARGET_LEVELS:
                plot_group_calendar_lines(g, m, lev)
    
    print("\n--- Generating 2D Time-Pressure Plots ---")
    all_cases = set()
    for g in GROUPS:
        for c in g["cases"]:
            all_cases.add(c["case_name"])
            
    for cname in sorted(list(all_cases)):
        for m in METRICS:
            plot_2d_section(cname, m)

    print("\n[BlockF] All Finished.")

if __name__ == "__main__":
    main_blockF()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockG — Hindcast Eddy Heat Flux (VTH3d) Verification
=====================================================
Target: VTH3d (Meridional Heat Flux), Zonal Mean, 45–75N Area Mean.
Reference: EHF_MERGED_45_75N_lev_time.nc (Your long-run truth).

Method:
  1. Load VTH3d from hindcast output.
  2. Clean dimensions (remove plev_2).
  3. Compute Zonal Mean -> 45-75N Weighted Mean.
  4. Compare with Reference.
"""

import os
import re
import shutil
from pathlib import Path
import numpy as np
import xarray as xr

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
HINDCAST_BASE_DIR = "/home/weiji/restart_exam/hindcast_data"

# 参考文件路径 (BlockA 生成的)
REF_EHF_PATH = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF/EHF_MERGED_45_75N_lev_time.nc"

OUT_DIR = "/home/weiji/test_code/plots/hindcast/EHF"
os.makedirs(OUT_DIR, exist_ok=True)


# ---------------------------------------------------------------------
# Scoring Functions
# ---------------------------------------------------------------------
def crps_ensemble(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    ens = np.asarray(ensemble)
    obs_arr = np.asarray(obs)
    if obs_arr.ndim == 1: obs_arr = obs_arr[None, :]
    
    diff_to_obs = np.abs(ens - obs_arr)
    term1 = diff_to_obs.mean(axis=0)
    diff_ij = np.abs(ens[:, None, ...] - ens[None, :, ...])
    term2 = 0.5 * diff_ij.mean(axis=(0, 1))
    return term1 - term2

def ensemble_spread(ensemble: np.ndarray) -> np.ndarray:
    return np.asarray(ensemble).std(axis=0, ddof=0)

def ensemble_bias(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    return np.asarray(ensemble).mean(axis=0) - np.asarray(obs)

def ensemble_mae(ensemble: np.ndarray, obs: np.ndarray) -> np.ndarray:
    return np.abs(np.asarray(ensemble) - np.asarray(obs)).mean(axis=0)


# ---------------------------------------------------------------------
# Core Logic: Process ONE VTH3d file
# ---------------------------------------------------------------------
def load_and_process_single_member(fpath: Path, member_id: int) -> xr.DataArray | None:
    """
    读取 VTH3d，清洗维度，计算 45-75N 平均。
    """
    try:
        ds = xr.open_dataset(fpath, decode_times=True)
        
        # 1. 变量名通常是 VTH3d
        vname = "VTH3d"
        if vname not in ds:
            # 备用方案，有时候可能叫 vT, VTP, etc.
            print(f"    [WARN] Variable '{vname}' not found in {fpath.name}")
            ds.close(); return None
        
        da = ds[vname] # (time, plev_2, lat, lon)

        # 2. 【维度清洗】杀掉 plev_2 (保留 plev 或重命名)
        #    你的例子里只有 plev_2，所以我们要把它重命名为 lev
        if "plev_2" in da.dims:
            if "plev" in da.dims:
                da = da.isel(plev_2=0, drop=True) # 两个都有，删一个
            else:
                da = da.rename({"plev_2": "plev"}) # 只有一个，改名
        
        if "ilev" in da.dims: da = da.isel(ilev=0, drop=True)

        # 3. 维度标准化
        rename_map = {}
        for d in da.dims:
            if d.lower().startswith("lon"): rename_map[d] = "lon"
            elif d.lower().startswith("lat"): rename_map[d] = "lat"
            elif d in ["lev_p", "level", "plev"]: rename_map[d] = "lev"
        
        if rename_map: da = da.rename(rename_map)

        # 4. 区域计算
        # 4a. 经向平均 (Zonal Mean)
        if "lon" in da.dims:
            da = da.mean(dim="lon")
        
        # 4b. 45-75N 加权平均
        if "lat" in da.dims:
            da_band = da.sel(lat=slice(45, 75))
            if da_band.sizes["lat"] == 0:
                raise ValueError("Empty lat slice 45-75N")
            w = np.cos(np.deg2rad(da_band.lat))
            da = da_band.weighted(w).mean("lat")

        # 5. 单位与垂直层处理
        #    确保垂直维叫 'lev' 且单位是 Pa
        if "lev" not in da.dims:
            # 试图找其他垂直维
            for d in da.dims:
                if "lev" in d or "press" in d:
                    da = da.rename({d: "lev"})
                    break
        
        if "lev" in da.dims:
            lev_vals = da.lev.values
            if np.nanmax(lev_vals) <= 2000.0: # hPa -> Pa
                da = da.assign_coords(lev=lev_vals * 100.0)
                da.lev.attrs["units"] = "Pa"

        # 6. 最终清洗 (Squeeze)
        da = da.squeeze().reset_coords(drop=True)
        
        ds.close()
        
        # Expand member
        da = da.expand_dims("member").assign_coords(member=[member_id])
        return da

    except Exception as e:
        print(f"    [Error] Processing {fpath.name}: {e}")
        return None


# ---------------------------------------------------------------------
# Loaders
# ---------------------------------------------------------------------
def load_case_ensemble(case_dir: Path) -> xr.DataArray | None:
    # 假设 VTH3d 在 case_dir/VTH3d/*.nc
    src_dir = case_dir / "VTH3d"
    if not src_dir.is_dir():
        # 有时候可能在根目录，或者叫其他名字，这里按你的描述
        return None
    
    files = sorted(src_dir.glob("*.nc"))
    if not files: return None

    print(f"\n[LOADER] Loading {case_dir.name} VTH3d ({len(files)} files)")

    da_list = []
    for idx, f in enumerate(files):
        da = load_and_process_single_member(f, idx + 1)
        if da is not None:
            da_list.append(da)

    if not da_list: return None

    try:
        ens = xr.concat(da_list, dim="member")
        if ens.ndim != 3:
            # 二次防御: (member, time, lev, 1) -> (member, time, lev)
            ens = ens.squeeze()
        print(f"    [LOADER] Final shape: {ens.shape}")
        return ens
    except Exception as e:
        print(f"    [Error] Concat failed: {e}")
        return None

def load_reference_slice(ref_path, case_name):
    if not os.path.exists(ref_path):
        raise FileNotFoundError(f"Missing Ref: {ref_path}")
        
    ds = xr.open_dataset(ref_path)
    var_name = list(ds.data_vars)[0]
    da_ref = ds[var_name] # (time, lev)
    
    # Parse Year-Month
    m = re.match(r"(\d{4})-(\d{2})", case_name)
    if not m: return None
    yr, mn = int(m.group(1)), int(m.group(2))
    
    start = f"{yr:04d}-{mn:02d}-01"
    end   = f"{yr+1:04d}-01-01"
    
    try:
        sl = da_ref.sel(time=slice(start, end))
        # Unit check
        if np.max(sl.lev.values) <= 2000.0:
            sl = sl.assign_coords(lev=sl.lev.values * 100.0)
        return sl
    except:
        return None


# ---------------------------------------------------------------------
# Compute Scores
# ---------------------------------------------------------------------
def compute_scores(case_name, ens_da, ref_path):
    ref_da = load_reference_slice(ref_path, case_name)
    if ref_da is None: return None
    
    ens_da, ref_da = xr.align(ens_da, ref_da, join="inner")
    if ens_da.time.size == 0: return None
    
    ens = ens_da.values
    obs = ref_da.values
    n_mem, n_t, n_lev = ens.shape
    
    CRPS = np.zeros((n_t, n_lev))
    Spread = np.zeros_like(CRPS)
    Bias = np.zeros_like(CRPS)
    MAE = np.zeros_like(CRPS)

    for k in range(n_lev):
        CRPS[:, k]   = crps_ensemble(ens[:,:,k], obs[:,k])
        Spread[:, k] = ensemble_spread(ens[:,:,k])
        Bias[:, k]   = ensemble_bias(ens[:,:,k], obs[:,k])
        MAE[:, k]    = ensemble_mae(ens[:,:,k], obs[:,k])

    ds_out = xr.Dataset(
        coords={"time": ens_da.time, "lev": ens_da.lev},
        data_vars={
            "CRPS": (("time", "lev"), CRPS),
            "spread": (("time", "lev"), Spread),
            "bias": (("time", "lev"), Bias),
            "mae": (("time", "lev"), MAE),
        },
        attrs={"case_name": case_name, "units": "K m/s"}
    )
    return ds_out


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------
def main_blockG():
    print("[BlockG] Starting VTH3d (EHF) Verification...")
    
    if not os.path.exists(REF_EHF_PATH):
        print("[Error] Reference file not found.")
        return

    cases = sorted(list(Path(HINDCAST_BASE_DIR).glob("0008-*")))
    
    for case_dir in cases:
        case_name = case_dir.name
        out_file = os.path.join(OUT_DIR, f"EHF_scores_{case_name}.nc")
        if os.path.exists(out_file): os.remove(out_file)
        
        ens = load_case_ensemble(case_dir)
        if ens is None: continue
        
        scores = compute_scores(case_name, ens, REF_EHF_PATH)
        if scores:
            scores.to_netcdf(out_file)
            print(f"  [Saved] {out_file}")

    print("\n[BlockG] Done.")

if __name__ == "__main__":
    main_blockG()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockH (Final Corrected) — Visualization for Eddy Heat Flux (VTH3d)
===================================================================
Target Levels: 10, 50, 100 hPa.
Units: K m/s.
Includes: 
  1. Lead-time Line Plots
  2. Calendar-time Line Plots
  3. 2D Time-Pressure Sections (with Smart Calendar Detection)
"""

import os
import re
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ----------------- 基础配置 -----------------
SCORES_ROOT_DIR = "/home/weiji/test_code/plots/hindcast/EHF"
FIG_OUT_DIR     = SCORES_ROOT_DIR
os.makedirs(FIG_OUT_DIR, exist_ok=True)

# 关注的层 (hPa)
TARGET_LEVELS = [10, 50, 100]

# 绘图时间范围 (Lead Days)
LEAD_START = 1
LEAD_END   = 150

# 指标配置
METRICS = [
    {"var": "CRPS",   "pretty": "CRPS",            "ylabel": "CRPS [K m/s]",       "cmap": "OrRd",   "min_zero": True},
    {"var": "bias",   "pretty": "Mean Error",      "ylabel": "Mean Error [K m/s]", "cmap": "RdBu_r", "min_zero": False},
    {"var": "spread", "pretty": "Ensemble Spread", "ylabel": "Spread [K m/s]",     "cmap": "YlGnBu", "min_zero": True},
]

# ----------------- 分组配置 (复用 BlockB/D/F) -----------------
GROUPS = [
    {
        "group_id": 0, "name": "group0_small_init", "title": "Group 0 – Small perturbation",
        "cases": [
            {"case_name": "0008-01", "label": "Jan small", "color": "C0"},
            {"case_name": "0008-02", "label": "Feb small", "color": "C1"},
            {"case_name": "0008-03", "label": "Mar small", "color": "C2"},
        ]
    },
    {
        "group_id": 1, "name": "group1_Feb_chem_vs_no", "title": "Group 1 – Feb: Chem vs Prescribe",
        "cases": [
            {"case_name": "0008-02",         "label": "Feb Chem",      "color": "C0"},
            {"case_name": "0008-02_NOCOUPL", "label": "Feb Prescribe", "color": "C3"},
        ]
    },
    {
        "group_id": 2, "name": "group2_Mar_chem_vs_no", "title": "Group 2 – Mar: Chem vs Prescribe",
        "cases": [
            {"case_name": "0008-03",         "label": "Mar Chem",      "color": "C0"},
            {"case_name": "0008-03_NOCOUPL", "label": "Mar Prescribe", "color": "C3"},
        ]
    },
    {
        "group_id": 3, "name": "group3_largepert", "title": "Group 3 – Large perturbation",
        "cases": [
            {"case_name": "0008-02_v2", "label": "Feb large", "color": "C0"},
            {"case_name": "0008-03_v2", "label": "Mar large", "color": "C1"},
            {"case_name": "0008-04_v2", "label": "Apr large", "color": "C2"},
        ]
    },
    {
        "group_id": 4, "name": "group4_Mar_diff_pert", "title": "Group 4 – Mar Init: Diff Pert",
        "cases": [
            {"case_name": "0008-03",    "label": "Small-pert", "color": "C0"},
            {"case_name": "0008-03_v2", "label": "Large-pert", "color": "C1"},
            {"case_name": "0008-03_v3", "label": "Q-pert",     "color": "C2"},
        ]
    },
    {
        "group_id": 5, "name": "group5_Qpert_diff_init", "title": "Group 5 – Q-pert",
        "cases": [
            {"case_name": "0008-02_v3", "label": "Feb Q-pert", "color": "C0"},
            {"case_name": "0008-03_v3", "label": "Mar Q-pert", "color": "C1"},
        ]
    },
    {
        "group_id": 6, "name": "group6_Feb_diff_pert", "title": "Group 6 – Feb Init: Diff Pert",
        "cases": [
            {"case_name": "0008-02",    "label": "Small-pert", "color": "C0"},
            {"case_name": "0008-02_v2", "label": "Large-pert", "color": "C1"},
            {"case_name": "0008-02_v3", "label": "Q-pert",     "color": "C2"},
        ]
    },
]

# ----------------- 核心工具 (智能日历检测) -----------------

def load_case_ds(case_name):
    # 注意文件名前缀是 EHF_scores_
    nc_path = os.path.join(SCORES_ROOT_DIR, f"EHF_scores_{case_name}.nc")
    if not os.path.exists(nc_path): return None
    try: 
        ds = xr.open_dataset(nc_path)
        return ds
    except Exception as e:
        # print(f"[Error] Open failed {case_name}: {e}")
        return None

def determine_anchor_year(ds, case_name):
    """
    决定绘图用的锚点年份。
    NoLeap (365天) -> 2001 (平年)
    Gregorian/Leap -> 2000 (闰年，适配 0008)
    """
    cal = ds.time.encoding.get('calendar', '') or ds.time.attrs.get('calendar', '')
    cal = str(cal).lower()
    
    anchor_year = 2001 # Default to NoLeap (Safe bet for WACCM)
    
    if 'noleap' in cal or '365_day' in cal:
        anchor_year = 2001
    elif 'gregorian' in cal or 'standard' in cal or 'proleptic' in cal:
        anchor_year = 2000 
    elif 'all_leap' in cal:
        anchor_year = 2000
        
    return anchor_year

def reconstruct_dates(ds, case_name):
    """重建日期对象，用于 Matplotlib 绘图"""
    match = re.search(r"(\d{4})-(\d{2})", case_name)
    start_month = int(match.group(2)) if match else 1
    
    anchor_year = determine_anchor_year(ds, case_name)
    start_date = pd.Timestamp(f"{anchor_year}-{start_month:02d}-01")
    
    n_steps = ds.sizes["time"]
    # 假设日数据
    dates = pd.date_range(start=start_date, periods=n_steps, freq='D')
    return dates

# ============================================================
# 1. Lead-time 折线图
# ============================================================

def plot_group_lead_lines(group_cfg, metric_cfg, level_hpa, fig_dir=FIG_OUT_DIR):
    var_name = metric_cfg["var"]
    target_pa = level_hpa * 100.0
    
    fig, ax = plt.subplots(figsize=(8, 5))
    has_data = False

    for cfg in group_cfg["cases"]:
        ds = load_case_ds(cfg["case_name"])
        if ds is None or var_name not in ds: continue
        try:
            da = ds[var_name].sel(lev=target_pa, method="nearest", tolerance=100)
            
            n_time = da.sizes["time"]
            eff_end = min(LEAD_END, n_time)
            idx_start = max(0, LEAD_START - 1)
            
            lead_days = np.arange(idx_start, eff_end)
            data_vals = da.isel(time=slice(idx_start, eff_end)).values
            
            ax.plot(lead_days, data_vals, label=cfg["label"], color=cfg.get("color"), lw=2, alpha=0.8)
            has_data = True
        except: pass
        finally: ds.close()

    if not has_data: plt.close(fig); return

    ax.set_xlabel("Lead Time (Days)", fontsize=12)
    ax.set_ylabel(metric_cfg["ylabel"], fontsize=12)
    ax.set_title(f"{group_cfg['title']}\n{metric_cfg['pretty']} @ {level_hpa} hPa (EHF)", fontweight='bold')
    ax.set_xlim(LEAD_START, LEAD_END)
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    if metric_cfg["min_zero"]: 
        bottom, top = ax.get_ylim()
        ax.set_ylim(0, top)
    
    fname = f"{var_name}_lead_{level_hpa}hPa_group{group_cfg['group_id']}.png"
    fig.savefig(os.path.join(fig_dir, fname), dpi=300)
    plt.close(fig)
    print(f"  -> Lead Plot: {fname}")

# ============================================================
# 2. Calendar-time 折线图
# ============================================================

def plot_group_calendar_lines(group_cfg, metric_cfg, level_hpa, fig_dir=FIG_OUT_DIR):
    var_name = metric_cfg["var"]
    target_pa = level_hpa * 100.0
    
    fig, ax = plt.subplots(figsize=(8, 5))
    has_data = False

    for cfg in group_cfg["cases"]:
        ds = load_case_ds(cfg["case_name"])
        if ds is None or var_name not in ds: continue
        try:
            da = ds[var_name].sel(lev=target_pa, method="nearest", tolerance=100)
            
            dates = reconstruct_dates(ds, cfg["case_name"])
            mask = (dates.month >= 1) & (dates.month <= 5) # Jan-May
            
            if np.any(mask):
                ax.plot(dates[mask], da.values[mask], label=cfg["label"], color=cfg.get("color"), lw=2, alpha=0.8)
                has_data = True
        except: pass
        finally: ds.close()

    if not has_data: plt.close(fig); return

    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    
    ax.set_xlabel("Calendar Time", fontsize=12)
    ax.set_ylabel(metric_cfg["ylabel"], fontsize=12)
    ax.set_title(f"{group_cfg['title']}\n{metric_cfg['pretty']} @ {level_hpa} hPa (Calendar EHF)", fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    if metric_cfg["min_zero"]: 
        bottom, top = ax.get_ylim()
        ax.set_ylim(0, top)

    fname = f"{var_name}_cal_{level_hpa}hPa_group{group_cfg['group_id']}.png"
    fig.savefig(os.path.join(fig_dir, fname), dpi=300)
    plt.close(fig)
    print(f"  -> Calendar Plot: {fname}")

# ============================================================
# 3. 2D Time-Pressure 填色图
# ============================================================

def plot_2d_section(case_name, metric_cfg, fig_dir=FIG_OUT_DIR):
    ds = load_case_ds(case_name)
    if ds is None: return

    var_name = metric_cfg["var"]
    if var_name not in ds: ds.close(); return

    try:
        # 准备数据
        Z = ds[var_name].transpose("lev", "time").values
        Y = ds.lev.values
        if np.max(Y) > 2000: Y = Y / 100.0 # Pa -> hPa
        
        dates = reconstruct_dates(ds, case_name)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Color Range
        if var_name == "bias":
            vmax = np.nanmax(np.abs(Z)) or 1
            vmin = -vmax
        else:
            vmax = np.nanmax(Z) or 1
            vmin = 0
            
        pcm = ax.pcolormesh(dates, Y, Z, shading="nearest", cmap=metric_cfg["cmap"], vmin=vmin, vmax=vmax)
        
        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(1000, 1) # Y轴范围
        ax.set_ylabel("Pressure (hPa)", fontsize=11)
        
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        
        ax.set_title(f"Case: {case_name} — {metric_cfg['pretty']} (EHF)", fontweight='bold')
        plt.colorbar(pcm, ax=ax, label=metric_cfg["ylabel"])
        
        fname = f"{var_name}_TimePress_{case_name}.png"
        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, fname), dpi=300)
        print(f"  -> 2D Plot: {fname}")
        
    except Exception as e:
        print(f"  [Error 2D] {case_name}: {e}")
    finally:
        ds.close(); plt.close(fig)

# ============================================================
# Main Entry
# ============================================================

def main_blockH():
    print(f"[BlockH] Output: {FIG_OUT_DIR}")
    
    # 1. Lead-time Plots
    print("\n--- Generating Lead-time Plots ---")
    for m in METRICS:
        for g in GROUPS:
            for lev in TARGET_LEVELS:
                plot_group_lead_lines(g, m, lev)
    
    # 2. Calendar-time Plots
    print("\n--- Generating Calendar-time Plots ---")
    for m in METRICS:
        for g in GROUPS:
            for lev in TARGET_LEVELS:
                plot_group_calendar_lines(g, m, lev)
    
    # 3. 2D Plots (Find all cases)
    print("\n--- Generating 2D Time-Pressure Plots ---")
    all_cases = set()
    for g in GROUPS:
        for c in g["cases"]:
            all_cases.add(c["case_name"])
            
    for cname in sorted(list(all_cases)):
        for m in METRICS:
            plot_2d_section(cname, m)

    print("\n[BlockH] All Finished.")

if __name__ == "__main__":
    main_blockH()